# Clinical Mortality Prediction Using the SUPPORT2 Dataset

**Organized research notebook — Cox regression, diagnostics, calibration, and Random Survival Forest**

**Navigation:** Open the **Table of contents** icon in the left sidebar of Google Colab. Expand the nine numbered sections to jump straight to any stage, table, or figure.

**Status:** The notebook contains previously executed Cox, stratified Cox, RSF and bootstrap analyses through Stage 21. These are historical saved outputs, not an independent end-to-end reproduction. Prediction-time validity, predictor availability, and external validation remain unresolved.

**Data:** UCI SUPPORT2, repository dataset ID 880, 9,105 patients. This is secondary research using a public clinical dataset, not a clinically validated decision tool. **Keep the original `.ipynb` as the unchanged research archive.**

**Important:** The cleaned notebook consolidates restoration and troubleshooting cells. Some cached plots and text contain legacy figure references; regenerate the relevant figures and reconcile the captions before journal submission. Saved outputs are historical; after opening in Colab, run cells top-to-bottom to regenerate results. A complete end-to-end run has not yet been independently verified.


## 1. Research Overview

**Goal:** Examine 180-day mortality prediction in the SUPPORT2 cohort using Kaplan–Meier estimation, penalized Cox regression, disease-group-stratified Cox regression, and a Random Survival Forest (RSF).

**Study design:** Retrospective analysis; stratified random 80/20 training/testing split (`random_state=42`). Primary outcome is observed death within 180 days, with survival follow-up administratively truncated at 180 days.

**Performance measures:** Survival concordance, test ROC-AUC, Brier score and risk calibration. The simple binary ROC-AUC and Brier score at day 180 are appropriate only when patients' 180-day outcomes are fully observed; the notebook checks for early censoring before evaluating them.

**Research limitations:** Observational associations are not causal; the dataset has a historical origin, and the work does not constitute external clinical validation. The Cox PH assumption was challenged for several disease-group effects; disease-group stratification is investigated as a partial remedy.

### Table and figure index (clean numbering)

| Item | What it shows | Section |
|---|---|---|
| Figure 1 | Kaplan–Meier survival curve | 2 |
| Table 1 / Figure 2 | Exploratory Cox hazard ratios / forest plot | 3 |
| Table 2 / Figure 3 | Initial proportional hazards tests | 4 |
| Table 3 / Figure 4 | FDR-adjusted PH diagnostics / colour-coded plot | 4 |
| Table 4 / Figure 5 | Stratified Cox PH tests / chart | 4 |
| Figure 6 | Scaled Schoenfeld residual trends | 4 |
| Table 5 / Figure 7 | Held-out Cox model evaluation / AUC comparison | 5 |
| Figures 8–9 | ROC and calibration curves | 5 |
| Table 6 / Figure 10 | Calibration by decile / error | 5 |
| Table 7 / Figure 11 | Quantitative calibration / comparison | 5 |
| Table 8 / Figure 12 | RSF predictor audit / completeness | 6 |
| Table 9 / Figure 13 | Survival outcome validation / distribution | 6 |
| Table 10 / Figure 14 | RSF concordance and comparison chart | 6 |
| Table 11 | Cox versus RSF test-set comparison | 7 |
| Figure 15 | Three-model ROC-AUC and Brier comparison | 7 |
| Tables 12–13 / Figure 16 | Bootstrap uncertainty and paired comparisons | 7 |

**Changes from original:** Original figure identifiers 13 and 14 used for exploratory Cox/PH plots have been reassigned; duplicate figure 10 and table 10 labels have been resolved in this new notebook only. The original archive retains its numbering for audit purposes.


## 2. Data Preparation

The data-loading, survival-time restoration, outcome definition, and predictor audits are now in one sequence. **Do not insert later “restore” cells here:** they are replaced by the consolidated steps below.


### Run once — Environment and output directory

Install dependencies before importing or executing modelling cells. Colab downloads the original UCI data whenever a fresh runtime begins. Package installations can take a few minutes.


In [ ]:
%pip -q install ucimlrepo lifelines scikit-survival statsmodels

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR = Path("support2_results")
OUT_DIR.mkdir(exist_ok=True)
print("Outputs will be saved in:", OUT_DIR.resolve())


### Stage 1 — Load the SUPPORT2 dataset


In [ ]:
from ucimlrepo import fetch_ucirepo

support2 = fetch_ucirepo(id=880)
X, y = support2.data.features, support2.data.targets
df = pd.concat([X, y], axis=1)
df = df.loc[:, ~df.columns.duplicated()].copy()

# UCI features/targets omit the original follow-up field d.time.
original = support2.data.original
assert df.index.equals(original.index), "Row indices differ; verify the source before joining."
assert (df['death'].to_numpy() == original['death'].to_numpy()).all(), "Death outcome mismatch."
df['d.time'] = original['d.time'].copy()

assert len(df) == 9105, f"Unexpected cohort size: {len(df)}"
assert df.index.is_unique
print("Dataset loaded:", df.shape)
print("Restored survival follow-up: d.time")
display(df.head())


### Stage 1: Dataset Acquisition and Initial Verification — Interpretation

The SUPPORT2 dataset was successfully retrieved from the UCI Machine Learning Repository for the analysis of clinical mortality and survival outcomes among seriously ill hospitalised patients.

The dataset contains **9,105 patient records and 46 variables**, following restoration of the original survival follow-up variable (`d.time`). The available information includes demographic characteristics, clinical diagnoses, comorbidities, physiological measurements, and survival-related variables.

Successful data acquisition and restoration of follow-up time provide the foundation for constructing the 180-day mortality outcome and conducting subsequent survival analysis.

At this stage, the dataset has been loaded and its structure verified. Further data quality assessment, outcome construction, and model development are required before drawing conclusions about mortality predictors.

**Conclusion:** Stage 1 confirms successful dataset acquisition and the availability of the information needed to proceed with data preparation and survival modelling.

### Stage 2 — Audit survival outcomes and follow-up


In [ ]:

# STAGE 2: VERIFY SURVIVAL OUTCOMES

print("DEATH INDICATOR")
print(df["death"].value_counts(dropna=False))

print("\nFOLLOW-UP TIME")
print(df["d.time"].describe())

print("\nMISSING OUTCOME DATA")
print(df[["death", "d.time"]].isna().sum())

print("\nINVALID FOLLOW-UP VALUES")
print((df["d.time"] <= 0).sum())

print("\nDEATH INDICATOR BY FOLLOW-UP PERIOD")
print(pd.crosstab(
    df["death"],
    df["d.time"] <= 180,
    dropna=False
))


### Stage 2: Survival Data Quality Assessment — Interpretation

The preliminary survival data assessment confirmed that the SUPPORT2 dataset contained 9,105 patient observations with complete information on mortality status and follow-up duration.

Of these patients, 6,201 experienced death during the recorded follow-up period, while 2,904 had no recorded death. A total of 4,265 deaths occurred within 180 days, representing approximately 46.8% of the study population. An additional 1,936 deaths occurred after 180 days.

The mean follow-up duration was 478.45 days, compared with a median of 233 days, indicating a right-skewed distribution of follow-up times. Recorded follow-up ranged from 3 to 2,029 days.

No missing mortality indicators, missing follow-up durations, or non-positive follow-up values were identified. The cross-tabulation also indicated no censoring before the 180-day endpoint.

**Conclusion:** The mortality and follow-up variables passed the initial completeness and validity checks. These findings support the construction of the 180-day mortality outcome for subsequent survival modelling. However, additional assessment of the predictors and modelling assumptions remains necessary.

In [ ]:
assert df[['death','d.time']].notna().all().all(), "Missing event or follow-up values."
assert df['death'].isin([0,1]).all(), "Unexpected death coding."
assert (df['d.time'] > 0).all(), "Invalid nonpositive survival time."
print("Outcome and follow-up checks passed.")


### Stage 3 — Construct the 180-day survival outcome


In [ ]:

# STAGE 3: DEFINE 180-DAY SURVIVAL OUTCOME

import numpy as np
import pandas as pd

# Preserve the original dataset
survival_df = df.copy()

# Set the prediction horizon
horizon = 180

# Construct survival time and event indicator
survival_df["time_180"] = np.minimum(
    survival_df["d.time"], horizon
)

survival_df["event_180"] = (
    (survival_df["death"] == 1) &
    (survival_df["d.time"] <= horizon)
).astype(int)

# Summarise the 180-day outcome
n = len(survival_df)
events = survival_df["event_180"].sum()

early_censored = (
    (survival_df["death"] == 0) &
    (survival_df["d.time"] < horizon)
).sum()

print("Total patients:", n)
print("Observed deaths within 180 days:", events)
print("Censored before 180 days:", early_censored)
print("No observed death within 180 days:", n - events)

print("\nOutcome distribution:")
print(survival_df["event_180"].value_counts())

print("\nFirst five observations:")
display(
    survival_df[
        ["death", "d.time", "time_180", "event_180"]
    ].head()
)


**Interpretation — Stage 3: Construction and Descriptive Analysis of the 180-Day Mortality Outcome**

The SUPPORT2 dataset comprised 9,105 seriously ill hospitalised patients. Mortality status and follow-up duration were examined to establish the primary survival outcome.

A total of 6,201 patients (68.1%) experienced death during the entire recorded follow-up period. However, because the study focuses on 180-day mortality, follow-up time was administratively restricted to 180 days.

Following this restriction, 4,265 patients (46.8%) experienced death within 180 days, while 4,840 patients (53.2%) had no recorded death within the specified period.

No patients were censored before 180 days, and neither the mortality indicator nor follow-up duration contained missing observations.

These findings provide the basis for subsequent Kaplan–Meier survival estimation, Cox proportional hazards regression, and Random Survival Forest modelling.

**Limitation:** These descriptive results do not identify the clinical determinants of mortality or establish whether one predictive model performs better than another. In addition, the interpretation assumes that the recorded survival times and death indicators accurately capture mortality and follow-up.


### Stage 4 — Kaplan–Meier survival analysis (Figure 1)


In [ ]:

# STAGE 4: KAPLAN-MEIER SURVIVAL ANALYSIS


from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

kmf.fit(
    durations=survival_df["time_180"],
    event_observed=survival_df["event_180"],
    label="Overall patient survival"
)

# Kaplan-Meier survival curve
fig, ax = plt.subplots(figsize=(9, 5))

kmf.plot_survival_function(
    ax=ax,
    ci_show=True
)

ax.set_title(
    "Figure 1. Kaplan–Meier Survival Curve: SUPPORT2"
)
ax.set_xlabel("Days since study entry")
ax.set_ylabel("Estimated survival probability")
ax.set_xlim(0, 180)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Estimated 180-day survival
survival_180 = kmf.predict(180)

print(
    "180-day survival probability:",
    round(survival_180, 4)
)
print(
    "180-day mortality probability:",
    round(1 - survival_180, 4)
)


**Interpretation — Stage 4: Kaplan–Meier Survival Analysis**

The Kaplan–Meier method was applied to estimate survival probabilities over a 180-day follow-up period among 9,105 seriously ill hospitalised patients included in the SUPPORT2 dataset.

The survival curve demonstrated a progressive reduction in survival probability throughout the observation period. The decline was particularly pronounced during the early stages of follow-up, suggesting a concentration of mortality events shortly after study entry.

The estimated survival probability at 180 days was approximately 53.2%, corresponding to a cumulative mortality probability of 46.8%. These findings are consistent with the 4,265 deaths observed during the first 180 days.

The curve became progressively flatter over time, indicating that the rate of decline in survival probability was less pronounced during later follow-up.

**Limitation:** The Kaplan–Meier analysis describes overall survival but does not account for differences in patient characteristics, disease severity, or other potential prognostic factors. Consequently, it cannot establish which clinical variables independently predict mortality.

Further analyses using Cox proportional hazards regression and Random Survival Forests will investigate the relationships between baseline patient characteristics and mortality outcomes.


### Stage 5 — Variable documentation


In [ ]:

# STAGE 5: REVIEW VARIABLES BEFORE MODELLING

variable_info = support2.variables.copy()

display(variable_info)

print("\nTotal columns in working dataset:")
print(len(df.columns))

print("\nAvailable variable names:")
for column in df.columns:
    print(column)


### Interpretation — Stage 5: Variable Documentation and Data Dictionary

**Objective**

The purpose of this stage was to examine the structure, definitions, and measurement characteristics of variables available in the SUPPORT2 dataset before selecting predictors for survival modelling.

**Findings**

The dataset documentation identified clinical and demographic variables, including patient age, sex, primary disease category, number of comorbidities, education, income, diabetes status, dementia status, and cancer status.

The documentation also distinguished outcome-related variables, such as mortality status and follow-up duration, from potential explanatory variables.

The available variables represented different measurement scales, including continuous, integer, binary, and categorical data. These distinctions are important because categorical predictors require appropriate encoding before inclusion in statistical and machine-learning models.

The metadata identified potential missingness in several variables. However, metadata descriptions alone do not establish the actual number or proportion of missing observations in the working dataset. These must be assessed empirically during data preparation.

**Methodological Implications**

Variable selection should be guided by clinical relevance, measurement quality, and availability at the intended prediction time.

Mortality outcomes and follow-up duration will be used to define the survival response rather than as predictors. Care must also be taken to prevent data leakage from outcome-related or post-baseline information.

**Limitation**

The data dictionary provides descriptions of available variables but does not establish their statistical associations with mortality, predictive contributions, or suitability for inclusion in the final models.

**Conclusion**

This stage established the foundation for systematic predictor selection and appropriate data preprocessing. The subsequent analysis will assess missing values, predictor distributions, and data quality before model estimation.

### Stage 6 — Baseline missingness and entry-timing audit


In [ ]:

# STAGE 6: AUDIT STUDY ENTRY AND PREDICTOR TIMING

import pandas as pd

print("STUDY ENTRY DAY DISTRIBUTION")
print(survival_df["hday"].describe())

print("\nFREQUENCY BY HOSPITAL DAY OF ENTRY")
display(
    survival_df["hday"]
    .value_counts(dropna=False)
    .sort_index()
    .head(30)
)

print("\nENTRY TIMING CATEGORIES")

entry_timing = pd.cut(
    survival_df["hday"],
    bins=[-float("inf"), 2, 3, float("inf")],
    labels=["Before day 3", "Day 3", "After day 3"]
)

display(entry_timing.value_counts(dropna=False))

# Candidate study-entry predictors
candidates = [
    "age", "sex", "dzgroup", "num.co",
    "diabetes", "dementia", "ca",
    "edu", "income"
]

print("\nCANDIDATE PREDICTOR MISSINGNESS")

audit = pd.DataFrame({
    "Missing_N": survival_df[candidates].isna().sum(),
    "Missing_Percent": (
        survival_df[candidates].isna().mean() * 100
    ).round(2)
})

display(audit.sort_values(
    "Missing_Percent", ascending=False
))


**Interpretation — Stage 6: Assessment of Missing Data Among Candidate Predictors**

Missing-data patterns were examined for nine candidate predictors in the SUPPORT2 dataset (N = 9,105).

Seven variables—age, sex, disease group, number of comorbidities, diabetes, cancer status, and dementia—had complete observations.

In contrast, income had 2,982 missing values (32.75%), while education had 1,634 missing values (17.95%).

The substantial missingness in these two variables requires further investigation because complete-case analysis may reduce statistical precision and introduce selection bias if missingness is associated with patient characteristics or outcomes.

The final treatment of missing data will be determined after evaluating missingness patterns and establishing the model development and validation strategy. Any imputation procedures will be fitted using training data to prevent information leakage.


In [ ]:
# STAGE 6: VISUALISATION OF MISSING DATA
# Figure 2: Missingness among candidate predictors

import pandas as pd
import matplotlib.pyplot as plt

predictors = [
    "age", "sex", "dzgroup", "num.co",
    "diabetes", "ca", "dementia",
    "income", "edu"
]

missing_pct = (
    df[predictors]
    .isna()
    .mean()
    .mul(100)
    .sort_values()
)

# Custom colors for easier identification
color_map = {
    "income": "#d73027",     # dark red
    "edu": "#fc8d59",        # blue
    "age": "#66c2a5",
    "sex": "#66c2a5",
    "dzgroup": "#66c2a5",
    "num.co": "#66c2a5",
    "diabetes": "#66c2a5",
    "ca": "#66c2a5",
    "dementia": "#66c2a5"
}

bar_colors = [color_map[var] for var in missing_pct.index]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    missing_pct.index,
    missing_pct.values,
    color=bar_colors
)

for bar, value in zip(bars, missing_pct.values):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center",
        fontsize=10
    )

ax.set_xlim(0, max(40, missing_pct.max() + 7))
ax.set_xlabel("Missing observations (%)")
ax.set_ylabel("Candidate predictors")
ax.set_title(
    "Figure 2. Missing Data Among Candidate Predictors",
    fontsize=13,
    fontweight="bold"
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(
    "Figure_2_Missing_Data.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Figure 2 saved successfully.")

### Interpretation — Figure 2: Missing Data Among Candidate Predictors

Figure 2 illustrates the proportion of missing observations among nine candidate predictors in the SUPPORT2 dataset (N = 9,105).

The results reveal substantial differences in data completeness. Income had the highest proportion of missing observations (32.75%), followed by education (17.95%). In contrast, the remaining seven predictors—age, sex, disease group, number of comorbidities, diabetes, cancer status, and dementia—had no missing observations.

These findings suggest that most selected clinical and demographic predictors have complete data, supporting their use in subsequent survival modelling. However, the substantial missingness in income and education requires careful consideration.

Excluding patients with missing observations could reduce the effective sample size and potentially introduce selection bias if missingness is associated with patient characteristics or mortality outcomes.

**Limitation:** The visualisation describes the extent of missingness but does not establish whether the data are missing completely at random (MCAR), missing at random (MAR), or missing not at random (MNAR).

**Conclusion:** The findings support prioritising complete baseline predictors while investigating appropriate strategies for handling income and education. Any imputation must be fitted using training data only to avoid information leakage.

### Stage 7 — Examine hospital-day entry timing


In [ ]:

# STAGE 7: STUDY-ENTRY TIMING ASSESSMENT

import pandas as pd

print("STUDY ENTRY DAY SUMMARY")
print(survival_df["hday"].describe())

print("\nSTUDY ENTRY DAY FREQUENCIES")
display(
    survival_df["hday"]
    .value_counts(dropna=False)
    .sort_index()
)

# Group patients by entry timing
timing = pd.cut(
    survival_df["hday"],
    bins=[-float("inf"), 2, 3, float("inf")],
    labels=[
        "Before day 3",
        "Day 3",
        "After day 3"
    ]
)

print("\nENTRY TIMING DISTRIBUTION")
display(
    pd.DataFrame({
        "Count": timing.value_counts().sort_index(),
        "Percent": (
            timing.value_counts().sort_index()
            / len(survival_df) * 100
        ).round(2)
    })
)

print("\nDEATHS BY ENTRY TIMING")
display(
    pd.crosstab(
        timing,
        survival_df["event_180"],
        margins=True
    )
)


**Interpretation — Stage 7: Study-Entry Timing and Implications for Predictor Selection**

Study-entry timing was examined among 9,105 patients in the SUPPORT2 dataset. The median hospital day of study entry was 1, while the mean was 4.40 days (SD = 9.13), indicating substantial variation in entry timing.

Most patients (70.77%) entered the study before hospital day 3, while 4.37% entered on day 3 and 24.85% entered after day 3.

The observed 180-day mortality proportions were 42.58%, 44.72%, and 59.35% for patients entering before day 3, on day 3, and after day 3, respectively.

These findings indicate that mortality outcomes varied across study-entry groups. However, the differences are descriptive and may reflect variations in clinical severity, patient selection, or other confounding factors.

For the primary prediction analysis, study entry will serve as the time origin. Predictor eligibility will be determined according to whether measurements were available at or before study entry.

Variables recorded after the prediction time will be excluded from the primary models to minimise temporal data leakage.

**Limitation:** The timing analysis does not establish whether all candidate clinical measurements were available at study entry. The prediction framework therefore requires explicit verification of predictor definitions before model development.


In [ ]:

# STAGE 7: STUDY-ENTRY TIMING VISUALISATION
# Figure 3: Distribution of Hospital-Day Entry

import matplotlib.pyplot as plt

timing_labels = [
    "Before day 3",
    "Day 3",
    "After day 3"
]

counts = [
    (df["hday"] < 3).sum(),
    (df["hday"] == 3).sum(),
    (df["hday"] > 3).sum()
]

colors = ["#247BA0", "#F4A261", "#2A9D8F"]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    timing_labels,
    counts,
    color=colors,
    width=0.6
)

for bar, count in zip(bars, counts):
    percentage = count / len(df) * 100

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f"{count:,}\n({percentage:.2f}%)",
        ha="center",
        fontsize=11
    )

ax.set_ylim(0, max(counts) * 1.18)
ax.set_ylabel("Number of patients")
ax.set_xlabel("Hospital day of study entry")
ax.set_title(
    "Figure 3. Distribution of Study-Entry Timing",
    fontsize=13,
    fontweight="bold"
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(
    "Figure_3_Study_Entry_Timing.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


Figure 3 illustrates the timing of study entry among 9,105 seriously ill hospitalised patients in the SUPPORT2 dataset.

Most patients (70.77%) entered the study before hospital day 3, while 4.37% entered on day 3 and 24.85% entered after day 3. The median entry day was 1, compared with a mean of 4.40 days, indicating a positively skewed distribution.

The findings suggest that although most patients were recruited early during hospitalisation, a substantial proportion entered later.

**Methodological implication:** Variation in hospital-day entry may be relevant to predictor selection and interpretation. Hospital-day entry should not automatically be treated as baseline disease severity, and the survival time origin must be defined consistently.

**Limitation:** These descriptive statistics do not establish whether later study entry is associated with mortality risk.

**Conclusion:** Study-entry timing should be considered during model specification and sensitivity analysis to avoid inappropriate interpretation of survival outcomes.

In [ ]:

# SUPPORT2 — PREDICTION-TIME AND OUTCOME AUDIT
import numpy as np
import pandas as pd

audit_df = survival_df.copy()

required = ["death", "d.time", "hday", "event_180", "time_180"]
assert set(required).issubset(audit_df.columns)

print("PATIENTS:", len(audit_df))

print("\nSTUDY ENTRY DAY (hday)")
print(audit_df["hday"].describe())

print("\nENTRY TIMING")
print(pd.Series({
    "Before hospital day 3": (audit_df["hday"] < 3).sum(),
    "Hospital day 3": (audit_df["hday"] == 3).sum(),
    "After hospital day 3": (audit_df["hday"] > 3).sum(),
    "Missing hday": audit_df["hday"].isna().sum()
}))

expected_event = (
    (audit_df["death"] == 1) &
    (audit_df["d.time"] <= 180)
).astype(int)

expected_time = np.minimum(audit_df["d.time"], 180)

print("\nOUTCOME VALIDATION")
print("Event coding matches:",
      expected_event.equals(audit_df["event_180"]))

print("Time coding matches:",
      np.allclose(expected_time, audit_df["time_180"]))

print("Early censored:",
      ((audit_df["death"] == 0) &
       (audit_df["d.time"] < 180)).sum())

print("\nMORTALITY BY STUDY ENTRY PERIOD")
timing = pd.cut(
    audit_df["hday"],
    bins=[-np.inf, 2, 3, np.inf],
    labels=["Before day 3", "Day 3", "After day 3"]
)

print(pd.crosstab(
    timing,
    audit_df["event_180"],
    margins=True
))

print("\nAUDIT NOTE:")
print("Coding checks do not establish the clinical "
      "time origin or predictor availability. "
      "Verify these in the SUPPORT2 data dictionary.")


### Stage 8 — Select and audit the baseline predictors


In [ ]:

# STAGE 8: BASELINE PREDICTOR AUDIT

import pandas as pd

candidate_predictors = [
    "age", "sex", "dzgroup",
    "num.co", "diabetes",
    "dementia", "ca", "hday"
]

# Check that all variables exist
missing_columns = [
    c for c in candidate_predictors
    if c not in survival_df.columns
]

assert not missing_columns, (
    f"Columns not found: {missing_columns}"
)

# Create a separate candidate dataset
baseline_df = survival_df[
    ["time_180", "event_180"] +
    candidate_predictors
].copy()

print("BASELINE DATASET DIMENSIONS")
print(baseline_df.shape)

print("\nVARIABLE TYPES")
display(baseline_df.dtypes)

print("\nMISSING VALUES")
display(baseline_df.isna().sum())

print("\nCATEGORICAL VARIABLE DISTRIBUTIONS")
for col in ["sex", "dzgroup",
            "diabetes", "dementia", "ca"]:
    print(f"\n{col}")
    print(baseline_df[col].value_counts(
        dropna=False
    ))


### Interpretation — Stage 8: Baseline Predictor Audit

The baseline predictor audit examined eight preselected clinical and demographic variables among 9,105 seriously ill hospitalised patients in the SUPPORT2 dataset.

The sample comprised 5,125 male patients (56.3%) and 3,980 female patients (43.7%). Diabetes was recorded in 1,778 patients (19.5%), while dementia was recorded in 296 patients (3.3%).

Regarding cancer status, 5,995 patients (65.8%) had no cancer, 1,858 (20.4%) had metastatic cancer, and 1,252 (13.8%) were classified as having non-metastatic cancer.

These findings demonstrate considerable variation in the prevalence of baseline clinical characteristics, supporting their examination as potential mortality predictors.

**Methodological consideration:** Categorical predictors, particularly diagnosis group and cancer status, require appropriate encoding before model estimation. Predictor suitability must also be evaluated according to clinical relevance, measurement timing, and the intended prediction outcome.

**Conclusion:** The audit establishes the distributions of selected baseline predictors and provides a foundation for subsequent Cox regression and Random Survival Forest modelling. However, descriptive frequencies alone do not establish predictive importance or associations with mortality.


In [ ]:
# STAGE 8 VISUAL: BASELINE PREDICTOR DISTRIBUTIONS

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ---------- Helper function ----------
def plot_categorical_distribution(ax, series, title, xlabel, color_map=None):
    counts = series.value_counts(dropna=False)
    total = counts.sum()
    perc = (counts / total * 100).round(1)

    labels = [str(x) for x in counts.index]

    # assign colors
    if color_map is not None:
        colors = [color_map.get(label, "gray") for label in labels]
    else:
        colors = None

    bars = ax.bar(labels, counts.values, color=colors)

    # add count and percent labels
    for bar, count, pct in zip(bars, counts.values, perc.values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(counts.values)*0.01,
            f"{count}\n({pct}%)",
            ha='center',
            va='bottom',
            fontsize=9
        )

    ax.set_title(title, fontsize=12, weight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of patients")
    ax.tick_params(axis='x', rotation=20)


# ---------- Prepare variables ----------
# Make readable copies
sex_plot = df["sex"].copy()

diabetes_plot = df["diabetes"].map({0: "No", 1: "Yes"}).fillna(df["diabetes"].astype(str))
dementia_plot = df["dementia"].map({0: "No", 1: "Yes"}).fillna(df["dementia"].astype(str))

ca_plot = df["ca"].copy()
ca_plot = ca_plot.replace({
    "no": "No cancer",
    "yes": "Non-metastatic cancer",
    "metastatic": "Metastatic cancer"
})

# ---------- Colors ----------
sex_colors = {
    "male": "#4C78A8",
    "female": "#F58518"
}

yes_no_colors = {
    "No": "#72B7B2",
    "Yes": "#E45756"
}

ca_colors = {
    "No cancer": "#54A24B",
    "Non-metastatic cancer": "#EECA3B",
    "Metastatic cancer": "#E45756"
}

# ---------- Create 2x2 figure ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_categorical_distribution(
    axes[0, 0], sex_plot,
    "Panel A. Sex Distribution",
    "Sex",
    color_map=sex_colors
)

plot_categorical_distribution(
    axes[0, 1], diabetes_plot,
    "Panel B. Diabetes Status",
    "Diabetes",
    color_map=yes_no_colors
)

plot_categorical_distribution(
    axes[1, 0], dementia_plot,
    "Panel C. Dementia Status",
    "Dementia",
    color_map=yes_no_colors
)

plot_categorical_distribution(
    axes[1, 1], ca_plot,
    "Panel D. Cancer Status",
    "Cancer status",
    color_map=ca_colors
)

fig.suptitle("Figure 4. Distribution of Baseline Clinical Predictors", fontsize=15, weight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])

# save figure
plt.savefig(
    OUT_DIR / "Figure_4_Baseline_Predictor_Distributions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Interpretation — Figure 4: Distribution of Baseline Clinical Predictors

Figure 4 presents the distribution of four baseline demographic and clinical predictors among 9,105 patients in the SUPPORT2 dataset.

The sample comprised 5,125 male patients (56.3%) and 3,980 female patients (43.7%). Diabetes was recorded in 1,778 patients (19.5%), while 7,327 patients (80.5%) did not have diabetes.

Dementia was comparatively uncommon, affecting 296 patients (3.3%), whereas 8,809 patients (96.7%) did not have dementia.

Regarding cancer status, 5,995 patients (65.8%) had no cancer, 1,858 (20.4%) had metastatic cancer, and 1,252 (13.8%) were classified as having non-metastatic cancer.

These results demonstrate variation in the prevalence of clinical conditions across the study population.

**Methodological implication:** The unequal distributions, particularly for dementia, should be considered when evaluating model performance across patient subgroups. The prevalence of these conditions does not independently establish their predictive importance.

**Limitation:** These descriptive results do not demonstrate associations with 180-day mortality.

**Conclusion:** Figure 4 provides a baseline characterisation of the study population and supports the subsequent examination of clinical predictors in Cox regression and Random Survival Forest models.

In [ ]:

# STAGE 8 — FIGURE 5: DISEASE GROUP DISTRIBUTION

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

OUT_DIR = Path("support2_results")
OUT_DIR.mkdir(exist_ok=True)

counts = df["dzgroup"].value_counts().sort_values()

# Different colour for each disease group
colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(
    counts.index,
    counts.values,
    color=colors,
    edgecolor="white"
)

# Add count and percentage
total = counts.sum()

for bar, count in zip(bars, counts.values):
    percentage = count / total * 100

    ax.text(
        bar.get_width() + 35,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,} ({percentage:.1f}%)",
        va="center",
        fontsize=10
    )

ax.set_xlim(0, counts.max() * 1.22)

ax.set_title(
    "Figure 5. Distribution of Disease Groups",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Number of patients")
ax.set_ylabel("Primary disease group")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    OUT_DIR / "Figure_5_Disease_Group_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### Interpretation — Figure 5: Distribution of Disease Groups

Figure 5 illustrates the distribution of eight primary disease groups among 9,105 seriously ill patients in the SUPPORT2 dataset.

Acute respiratory failure or multiple organ system failure with sepsis represented the largest diagnostic category, comprising 3,515 patients (38.6%). Congestive heart failure accounted for 1,387 patients (15.2%), followed by chronic obstructive pulmonary disease with 967 patients (10.6%) and lung cancer with 908 patients (10.0%).

The remaining categories included multiple organ system failure with malignancy (7.8%), coma (6.5%), colon cancer (5.6%) and cirrhosis (5.6%).

**Methodological implication:** The unequal distribution of diagnostic categories should be considered during predictive modelling. Disease group may contribute to differences in baseline mortality risk, but the present descriptive analysis cannot establish the magnitude or direction of these associations.

**Limitation:** The figure describes the composition of the study population rather than mortality rates within each diagnostic category. Further modelling is required to evaluate disease-specific associations with survival.

**Conclusion:** The distribution highlights clinical heterogeneity within the SUPPORT2 population and supports the inclusion of primary disease group as a categorical predictor in subsequent survival models.

## 3. Model Development

The single split is reused by all models; categorical levels are encoded using the training data. Subsequent cells depend on the variables created in this order.


### Stage 9 — Training and testing split


In [ ]:

# STAGE 9: TRAINING AND TESTING SPLIT

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    baseline_df,
    test_size=0.20,
    random_state=42,
    stratify=baseline_df["event_180"]
)

print("TRAINING SAMPLE")
print("Number of patients:", len(train_df))
print("Deaths:", train_df["event_180"].sum())
print("Mortality (%):",
      round(train_df["event_180"].mean() * 100, 2))

print("\nTESTING SAMPLE")
print("Number of patients:", len(test_df))
print("Deaths:", test_df["event_180"].sum())
print("Mortality (%):",
      round(test_df["event_180"].mean() * 100, 2))

print("\nPATIENT OVERLAP")
print(len(set(train_df.index) & set(test_df.index)))


**Interpretation — Stage 9: Training and Testing Data Partition**

The analytical dataset comprised 9,105 patients and was randomly divided into training (80%) and testing (20%) subsets using stratified sampling based on the 180-day mortality indicator.

The training subset contained 7,284 patients, including 3,412 deaths (46.84%), while the testing subset comprised 1,821 patients, including 853 deaths (46.84%).

A fixed random seed of 42 was used to ensure reproducibility. No patient observations overlapped between the training and testing subsets.

The training dataset will be used for model development, including preprocessing and hyperparameter tuning. The testing dataset will be reserved for evaluating predictive performance using identical patient observations for both modelling approaches.

This partition provides an internal holdout evaluation rather than external validation.


In [ ]:

# STAGE 9 — FIGURE 6: TRAINING AND TESTING DISTRIBUTION

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("support2_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the existing training and testing datasets
samples = {
    "Training": train_df,
    "Testing": test_df
}

names = list(samples.keys())
sizes = [len(data) for data in samples.values()]

deaths = [
    int(data["event_180"].sum())
    for data in samples.values()
]

survivors = [
    size - death
    for size, death in zip(sizes, deaths)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PANEL A: Sample sizes
colors = ["#4C78A8", "#F58518"]

bars = axes[0].bar(
    names, sizes, color=colors, width=0.55
)

for bar, size in zip(bars, sizes):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        size + 100,
        f"{size:,}\n({size/sum(sizes)*100:.1f}%)",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

axes[0].set_ylim(0, max(sizes)*1.20)
axes[0].set_title(
    "Panel A. Training and Testing Sample Sizes",
    fontweight="bold"
)
axes[0].set_ylabel("Number of patients")

# PANEL B: Mortality distribution (100% stacked)
x = np.arange(len(names))

death_pct = np.array(deaths) / sizes * 100
survivor_pct = np.array(survivors) / sizes * 100

axes[1].bar(
    x, survivor_pct,
    color="#54A24B",
    label="No death within 180 days"
)

axes[1].bar(
    x, death_pct,
    bottom=survivor_pct,
    color="#E45756",
    label="Death within 180 days"
)

for i in range(len(names)):
    axes[1].text(
        x[i], survivor_pct[i]/2,
        f"{survivors[i]:,}\n({survivor_pct[i]:.1f}%)",
        ha="center", va="center",
        fontweight="bold"
    )

    axes[1].text(
        x[i],
        survivor_pct[i] + death_pct[i]/2,
        f"{deaths[i]:,}\n({death_pct[i]:.1f}%)",
        ha="center", va="center",
        fontweight="bold"
    )

axes[1].set_xticks(x)
axes[1].set_xticklabels(names)
axes[1].set_ylim(0, 100)
axes[1].set_ylabel("Percentage of patients")
axes[1].set_title(
    "Panel B. 180-Day Mortality Distribution",
    fontweight="bold"
)

axes[1].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    frameon=False
)

fig.suptitle(
    "Figure 6. Training and Testing Sample Distribution",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0.06, 1, 0.94])

plt.savefig(
    OUT_DIR / "Figure_6_Training_Testing_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### Interpretation — Figure 6: Training and Testing Sample Distribution

Figure 6 illustrates the allocation of 9,105 patients in the SUPPORT2 dataset into training and testing subsets using an 80:20 stratified random partition.

The training sample comprised 7,284 patients (80%), while the testing sample contained 1,821 patients (20%).

Within the training dataset, 3,412 patients (46.84%) experienced death within 180 days, compared with 853 patients (46.84%) in the testing dataset. The corresponding proportions without an observed death within 180 days were approximately 53.16% in both samples.

**Methodological implication:** The nearly identical mortality distributions demonstrate that stratification successfully preserved the outcome proportions. The training dataset will support model development, while the independent testing dataset will be reserved for evaluating predictive performance.

**Limitation:** Stratification ensures balance in the event indicator but does not necessarily guarantee comparable distributions of clinical predictors or follow-up durations. Furthermore, the testing sample provides internal holdout validation rather than external validation.

**Conclusion:** The 80:20 partition establishes a consistent foundation for comparing Cox proportional hazards regression and Random Survival Forest models using the same training and testing datasets.

### Stage 10 — Encode predictors and verify feature alignment


In [ ]:

# STAGE 10: ENCODE PREDICTORS FOR COX REGRESSION

import pandas as pd

categorical_cols = ["sex", "dzgroup", "ca"]

# Use training data to define encoding
train_cox = pd.get_dummies(
    train_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

# Apply matching columns to test data
test_cox = pd.get_dummies(
    test_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

test_cox = test_cox.reindex(
    columns=train_cox.columns,
    fill_value=0
)

print("Training dimensions:", train_cox.shape)
print("Testing dimensions:", test_cox.shape)

print("\nEncoded predictor names:")
print(train_cox.columns.tolist())

print("\nMissing values:")
print(train_cox.isna().sum().sum())

print("\nFirst five training observations:")
display(train_cox.head())


In [ ]:

print("STAGE 10: FINAL VERIFICATION")

print("Identical columns and order:",
      train_cox.columns.equals(test_cox.columns))

print("Training missing values:",
      train_cox.isna().sum().sum())

print("Testing missing values:",
      test_cox.isna().sum().sum())

print("Training indices aligned:",
      train_cox.index.equals(train_df.index))

print("Testing indices aligned:",
      test_cox.index.equals(test_df.index))

print("Non-numeric training columns:",
      train_cox.select_dtypes(
          include=["object", "category"]
      ).columns.tolist())

print("Non-numeric testing columns:",
      test_cox.select_dtypes(
          include=["object", "category"]
      ).columns.tolist())


### Research interpretation — Stage 10
The preprocessing procedure produced numerically encoded training and testing datasets suitable for Cox proportional hazards regression. Both datasets contained identical variables in the same order, with no missing values or non-numeric columns. Their patient indices were also correctly aligned.

These checks establish structural consistency before model estimation. They do not, by themselves, establish that the model's statistical assumptions are satisfied.

In [ ]:

# FIGURE 7 — TRAINING AND TESTING DATA QUALITY

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

output_dir = Path("support2_results")
output_dir.mkdir(exist_ok=True)

datasets = {
    "Training": train_cox,
    "Testing": test_cox
}

completeness = [
    data.notna().mean().mean() * 100
    for data in datasets.values()
]

colors = ["#2563EB", "#10B981"]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    datasets.keys(),
    completeness,
    color=colors,
    width=0.45
)

for bar, value in zip(bars, completeness):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.1f}%",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

ax.set_ylim(0, 110)
ax.set_ylabel("Data completeness (%)")
ax.set_title(
    "Figure 7. Encoded Dataset Completeness",
    fontsize=14,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(
    output_dir / "Figure_7_Encoded_Data_Completeness.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


Interpretation
Both encoded datasets achieved 100% completeness, with no missing values detected in either the training or testing matrix. Combined with the verified column alignment and numerical encoding, this demonstrates that the datasets satisfy the structural requirements for subsequent Cox regression estimation.

Limitation: Complete data does not necessarily imply clinically valid measurements, unbiased sampling, or correctly specified predictors.

### Stage 11 — Exploratory penalized Cox regression


In [ ]:

# STAGE 11: EXPLORATORY COX REGRESSION

from lifelines import CoxPHFitter
import pandas as pd

# Check for constant predictor columns
predictor_columns = [
    c for c in train_cox.columns
    if c not in ["time_180", "event_180"]
]

constant_columns = [
    c for c in predictor_columns
    if train_cox[c].nunique() <= 1
]

print("Constant columns:", constant_columns)

# Exclude constant columns, if present
cox_data = train_cox.drop(
    columns=constant_columns
).copy()

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.01)

cph.fit(
    cox_data,
    duration_col="time_180",
    event_col="event_180"
)

# Display results
print("\nCOX REGRESSION RESULTS")
cph.print_summary()

print("\nHAZARD RATIOS AND 95% CONFIDENCE INTERVALS")
display(
    cph.summary[
        [
            "coef",
            "exp(coef)",
            "exp(coef) lower 95%",
            "exp(coef) upper 95%",
            "p"
        ]
    ]
)

print("\nTRAINING CONCORDANCE INDEX")
print(round(cph.concordance_index_, 4))


#### Table 1 and Figure 2 — Adjusted hazard ratios and 95% confidence intervals


### Table 1 and Figure 2. Cox Regression Results and Adjusted Hazard Ratios

**Interpretation:** Table 1 presents the estimated regression coefficients, adjusted hazard ratios (HR), 95% confidence intervals and p-values obtained from the exploratory L2-penalized Cox proportional hazards model. Figure 2 provides a graphical representation of these associations.

Age was positively associated with mortality hazard (HR = 1.015, 95% CI: 1.012–1.017), corresponding to an approximately 1.5% higher estimated hazard for each additional year of age, holding other model predictors constant. Similarly, each additional comorbidity was associated with an approximately 11.1% higher hazard (HR = 1.111, 95% CI: 1.077–1.147).

The diabetes indicator showed a lower adjusted hazard (HR = 0.889, 95% CI: 0.808–0.978). This association should be interpreted cautiously and does not establish a protective effect of diabetes. Dementia and male sex showed confidence intervals including one, providing insufficient evidence of an association at the conventional 5% significance level.

Several diagnostic categories also demonstrated substantial differences in estimated mortality hazard relative to the omitted reference diagnosis group. These comparisons reflect differences between patient groups rather than causal effects of diagnoses.

The model achieved a training concordance index of 0.6719, indicating moderate discrimination. However, this represents performance on the training sample and does not establish generalizability to unseen patients.

Overall, the findings demonstrate the potential of Cox regression for examining mortality-related risk factors while highlighting the need for proportional hazards diagnostics and independent testing. Because penalization was applied, the reported confidence intervals and p-values should be treated as exploratory.


In [ ]:

# STAGE 11 — COLOUR-CODED COX REGRESSION FOREST PLOT
# Figure 2: Adjusted Hazard Ratios and 95% Confidence Intervals

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Extract results from the fitted Cox model
results = cph.summary.copy()

forest = pd.DataFrame({
    "Predictor": results.index,
    "HR": results["exp(coef)"],
    "CI_Lower": results["exp(coef) lower 95%"],
    "CI_Upper": results["exp(coef) upper 95%"],
    "p_value": results["p"]
}).reset_index(drop=True)

# Classify results for meaningful colour coding
def classify(row):
    if row["p_value"] >= 0.05:
        return "Not statistically significant"
    elif row["HR"] > 1:
        return "Higher hazard"
    else:
        return "Lower hazard"

forest["Category"] = forest.apply(classify, axis=1)

colors = {
    "Higher hazard": "#DC2626",
    "Lower hazard": "#059669",
    "Not statistically significant": "#94A3B8"
}

# Sort by hazard ratio
forest = forest.sort_values("HR").reset_index(drop=True)

fig, ax = plt.subplots(
    figsize=(11, max(7, len(forest) * 0.5))
)

y = np.arange(len(forest))

for i, row in forest.iterrows():
    color = colors[row["Category"]]

    ax.errorbar(
        row["HR"],
        i,
        xerr=[[
            row["HR"] - row["CI_Lower"],
        ], [
            row["CI_Upper"] - row["HR"],
        ]],
        fmt="o",
        color=color,
        ecolor=color,
        markersize=7,
        capsize=4,
        linewidth=1.8
    )

# Reference line: HR = 1
ax.axvline(
    x=1,
    color="#475569",
    linestyle="--",
    linewidth=1.5
)

ax.set_yticks(y)
ax.set_yticklabels(forest["Predictor"], fontsize=10)

ax.set_xscale("log")
ax.set_xlabel("Adjusted Hazard Ratio (log scale)", fontsize=11)
ax.set_ylabel("Clinical predictors", fontsize=11)

ax.set_title(
    "Figure 2. Adjusted Hazard Ratios for 180-Day Mortality",
    fontsize=14,
    fontweight="bold",
    pad=18
)

ax.grid(axis="x", linestyle=":", alpha=0.3)
ax.set_axisbelow(True)

from matplotlib.lines import Line2D

legend = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=colors[k], markersize=9, label=k)
    for k in colors
]

ax.legend(handles=legend, loc="best", fontsize=9)

plt.tight_layout()

# Save high-resolution figure
output_dir = Path("support2_results")
output_dir.mkdir(exist_ok=True)

plt.savefig(
    output_dir / "Figure_2_Cox_Hazard_Ratios.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figure 2 saved successfully.")


### Stage 12 — Interpret complete Cox regression results


In [ ]:

# STAGE 12: COMPLETE COX RESULTS

import pandas as pd

cox_full_results = cph.summary[
    [
        "coef",
        "exp(coef)",
        "exp(coef) lower 95%",
        "exp(coef) upper 95%",
        "p"
    ]
].copy()

cox_full_results.columns = [
    "Coefficient",
    "Hazard_Ratio",
    "CI_Lower",
    "CI_Upper",
    "P_Value"
]

print("COMPLETE COX REGRESSION RESULTS")
display(cox_full_results.round(4))

print("\nTRAINING CONCORDANCE INDEX")
print(round(cph.concordance_index_, 4))

print("\nOMITTED REFERENCE CATEGORIES")
for col in ["sex", "dzgroup", "ca"]:
    print(col, sorted(train_df[col].unique()))


**Interpretation — Stage 12: Cox Proportional Hazards Regression Results**

A penalised Cox proportional hazards regression model was fitted to the training dataset comprising 7,284 patients, including 3,412 observed deaths within 180 days. The model incorporated demographic characteristics, comorbidity measures, disease classifications, cancer status, and hospital day of study entry.

The model achieved a training concordance index of 0.6719, indicating moderate ability to discriminate between patients according to their observed survival outcomes. This estimate reflects apparent discrimination in the development sample and requires validation using the independent testing subset.

**Age and comorbidity burden:** Increasing age was associated with a higher mortality hazard (HR = 1.015, 95% CI: 1.012–1.017). Each additional recorded comorbidity was associated with an approximately 11.1% higher estimated hazard of death (HR = 1.111, 95% CI: 1.077–1.147), conditional on the other included variables.

**Diabetes and dementia:** Diabetes was associated with a lower adjusted mortality hazard (HR = 0.889, 95% CI: 0.808–0.978), although this finding should not be interpreted as evidence of a protective causal effect. Dementia showed no statistically significant association in this model (HR = 1.086, 95% CI: 0.910–1.296).

**Study-entry timing:** Later hospital day of study entry was associated with higher mortality hazard (HR = 1.015, 95% CI: 1.012–1.018). This association may reflect differences in disease severity or patient selection rather than a causal effect of delayed study entry.

**Disease categories:** Compared with the reference category of acute respiratory failure/multiple organ system failure with sepsis, the coma disease group exhibited a substantially higher estimated hazard (HR = 2.846, 95% CI: 2.527–3.205), while patients classified under congestive heart failure exhibited a lower adjusted hazard (HR = 0.446, 95% CI: 0.391–0.509).

**Cancer status:** Metastatic cancer was the omitted reference category in the Cox regression model. Consequently, the estimated hazard ratios for patients without cancer (HR = 0.769) and patients classified as having non-metastatic cancer (HR = 0.829) represent comparisons with the metastatic cancer group, after adjustment for the other model covariates. These associations should not be interpreted as causal effects.

**Model Limitations:** These findings are exploratory. Subsequent proportional hazards diagnostics identified evidence of non-proportionality for several disease-group predictors, indicating that their hazard ratios may change over time. Furthermore, overlapping clinical classifications, possible nonlinear relationships, and the use of penalised estimation require careful interpretation. The training concordance index does not establish external predictive validity. Therefore, additional model refinement and independent test-set evaluation are necessary before drawing definitive clinical prognostic conclusions.

## 4. Model Diagnostics

We first test the proportional hazards (PH) assumption, adjust for multiple comparisons, then fit and evaluate a disease-group-stratified Cox model. Some model diagnostics may print statistical warnings and should be interpreted in context.


### Stage 13 — Proportional hazards tests (Table 2, Figure 3)


In [ ]:
# STAGE 13: COLOUR-CODED PROPORTIONAL HAZARDS DIAGNOSTICS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from lifelines.statistics import proportional_hazard_test

# Calculate proportional hazards diagnostics
# using the fitted Cox model from Stage 11
ph_test = proportional_hazard_test(
    cph,
    cox_data,
    time_transform="rank"
)

ph_results = ph_test.summary.copy()

print("STAGE 13: PROPORTIONAL HAZARDS TEST RESULTS")
display(ph_results.round(5))


# Reuse the proportional hazards test from Stage 13
ph_plot = ph_test.summary.copy()

# The original Lifelines output uses the column "p"
ph_plot["P_Value"] = ph_plot["p"]

# Predictor names are stored in the index
ph_plot["Predictor"] = ph_plot.index

# Prevent log(0) when p-values are extremely small
ph_plot["Score"] = -np.log10(
    ph_plot["P_Value"].clip(lower=1e-300)
)

# Assign colours according to statistical evidence
ph_plot["Colour"] = np.where(
    ph_plot["P_Value"] < 0.05,
    "tomato",
    "mediumseagreen"
)

# Order predictors by strength of evidence
ph_plot = ph_plot.sort_values("Score", ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))

ax.barh(
    ph_plot["Predictor"],
    ph_plot["Score"],
    color=ph_plot["Colour"],
    edgecolor="black",
    alpha=0.85
)

# Significance threshold
ax.axvline(
    -np.log10(0.05),
    color="purple",
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("-log10(p-value)")
ax.set_ylabel("Clinical predictors")
ax.set_title(
    "Figure 3. Proportional Hazards Diagnostic Tests",
    fontsize=13,
    fontweight="bold"
)

ax.legend(
    handles=[
        Patch(color="tomato",
              label="Potential PH violation (p < 0.05)"),
        Patch(color="mediumseagreen",
              label="No significant evidence (p >= 0.05)")
    ],
    loc="lower right"
)

plt.tight_layout()

plt.savefig(
    OUT_DIR / "Figure03_PH_Diagnostics_Coloured.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figure 3 saved successfully.")


### Table 2 and Figure 3. Assessment of the Proportional Hazards Assumption

**Interpretation:** The proportional hazards assumption was evaluated using Schoenfeld residual-based statistical tests. Table 2 presents the test statistics and corresponding p-values, while Figure 3 illustrates the strength of evidence against proportional hazards for individual predictors.

The results indicate statistically significant departures from the proportional hazards assumption for several diagnostic categories, including congestive heart failure, chronic obstructive pulmonary disease, cirrhosis, colon cancer and coma. Cancer-status indicators also demonstrated significant violations. These findings suggest that the associations between these predictors and mortality hazard may vary over the follow-up period.

In contrast, age, diabetes, dementia and sex did not demonstrate statistically significant violations at the 5% significance level. However, nonsignificant tests do not establish that proportional hazards holds exactly.

Overall, the findings indicate that the conventional Cox model may not adequately represent the time-dependent associations of several diagnostic predictors. Consequently, interpreting all estimated hazard ratios as constant over time would be inappropriate. A stratified Cox model, allowing different baseline hazards across clinically relevant patient groups, will therefore be investigated as an alternative. Further diagnostics and evaluation on the held-out testing sample are required before selecting the final model.


**Interpretation — Stage 13: Proportional Hazards Assumption Diagnostics**

The proportional hazards assumption of the exploratory Cox regression model was evaluated using tests based on scaled Schoenfeld residuals, applying a rank transformation of survival time.

The analysis identified nine coefficients with statistically significant evidence of departures from proportional hazards at the 5% significance level. These included several disease-group indicators, cancer-status categories, and hospital day of study entry.

The strongest evidence of non-proportionality was observed for congestive heart failure (χ² = 52.10, p < 0.001), coma (χ² = 40.93, p < 0.001), and chronic obstructive pulmonary disease (χ² = 34.92, p < 0.001).

Other significant results included colon cancer, lung cancer, cirrhosis, cancer-status indicators, and hospital day of study entry.

In contrast, age, sex, number of comorbidities, diabetes, and dementia showed no statistically significant departures from proportional hazards in the reported tests.

These findings suggest that the constant hazard ratio assumption may be inappropriate for several predictors in the initial model. Consequently, the estimated hazard ratios should be interpreted cautiously, as they may not adequately represent associations across the entire 180-day follow-up period.

The results justify further graphical assessment and consideration of extended Cox models incorporating time-varying effects, stratification where appropriate, or alternative functional forms.

**Limitation:** The tests were conducted on an exploratory penalised Cox model. Statistical significance may be influenced by sample size, model specification, and multiple testing. The results should therefore be assessed alongside graphical diagnostics before determining the final modelling strategy.


### Stage 14 — Colour-coded PH diagnostics and FDR adjustment (Figure 4, Table 3)


In [ ]:

# STAGE 14: TABLE 3 — FDR-ADJUSTED PH TEST RESULTS

import pandas as pd
from statsmodels.stats.multitest import multipletests
from IPython.display import display

# Use the existing Stage 13 PH test results
ph_table = ph_test.summary.copy()

# Get original p-values
p_values = ph_table["p"].astype(float)

# Benjamini-Hochberg false discovery rate correction
reject, p_adjusted, _, _ = multipletests(
    p_values,
    alpha=0.05,
    method="fdr_bh"
)

# Construct Table 3
table3 = pd.DataFrame({
    "Predictor": ph_table.index,
    "Chi_Square": ph_table["test_statistic"].values,
    "Original_P": p_values.values,
    "FDR_Adjusted_P": p_adjusted,
    "FDR_Significant": reject
})

# Sort by adjusted p-value
table3 = table3.sort_values(
    "FDR_Adjusted_P"
).reset_index(drop=True)

print("TABLE 3: FDR-ADJUSTED PROPORTIONAL HAZARDS TESTS")

display(
    table3.style.format({
        "Chi_Square": "{:.4f}",
        "Original_P": "{:.6g}",
        "FDR_Adjusted_P": "{:.6g}"
    })
)

# Save Table 3
table3.to_csv(
    OUT_DIR / "Table3_FDR_Adjusted_PH_Tests.csv",
    index=False
)

print("Table 3 saved successfully.")


In [ ]:

# STAGE 14: COLOUR-CODED PH DIAGNOSTIC GRAPH

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Sort predictors by statistical significance
ph_plot = ph_results.sort_values("p").copy()

# Calculate negative log10 p-values
ph_plot["neg_log10_p"] = -np.log10(
    ph_plot["p"].clip(lower=1e-300)
)

# Assign colours based on p-value
bar_colors = np.where(
    ph_plot["p"] < 0.05,
    "#D65A4A",  # Red: potential violation
    "#258875"   # Green: not statistically significant
)

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(
    ph_plot.index,
    ph_plot["neg_log10_p"],
    color=bar_colors,
    edgecolor="white"
)

# Statistical significance threshold
ax.axvline(
    -np.log10(0.05),
    color="#555555",
    linestyle="--",
    linewidth=1.5
)

ax.invert_yaxis()

ax.set_title(
    "Figure 4. Proportional Hazards Assumption Diagnostics",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel(
    "-log10(p-value)",
    fontsize=12
)

ax.set_ylabel("Clinical predictors", fontsize=12)

# Meaningful legend
legend_items = [
    Patch(color="#D65A4A",
          label="Potential PH violation (p < 0.05)"),
    Patch(color="#258875",
          label="No significant evidence (p >= 0.05)")
]

ax.legend(handles=legend_items, loc="lower right")

ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()

plt.savefig(OUT_DIR / 'Figure04_PH_Diagnostics_Colour_Coded.png',
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### Interpretation — Stage 14: Proportional Hazards Diagnostics and False Discovery Rate Adjustment

**Method:** The proportional hazards (PH) assumption was further evaluated using the Schoenfeld residual-based diagnostic results obtained in Stage 13. To account for multiple statistical comparisons across 15 model coefficients, the Benjamini–Hochberg false discovery rate (FDR) procedure was applied. Statistical significance was assessed using an FDR-adjusted p-value threshold of 0.05.

**Results:** Table 3 shows that 9 of the 15 tested coefficients remained statistically significant after FDR adjustment, indicating persistent evidence of departures from the proportional hazards assumption.

The strongest evidence was observed for congestive heart failure (CHF; adjusted p = 7.92 × 10⁻¹²), coma (adjusted p = 1.18 × 10⁻⁹), COPD (adjusted p = 1.72 × 10⁻⁸), colon cancer (adjusted p = 4.02 × 10⁻⁷), and lung cancer (adjusted p = 5.70 × 10⁻⁶).

Additional statistically significant findings were observed for cancer status without cancer (adjusted p = 0.00079), cirrhosis (adjusted p = 0.00237), hospital day of study entry (adjusted p = 0.01592), and cancer status classified as yes (adjusted p = 0.02661).

In contrast, age, sex, diabetes, dementia, number of comorbidities, and multiple organ system failure with malignancy did not demonstrate statistically significant evidence of PH violations after adjustment.

**Graphical interpretation:** Figure 4 displays the strength of evidence against the proportional hazards assumption using negative log-transformed original p-values. Red bars represent predictors with unadjusted p-values below 0.05, while green bars represent predictors without statistically significant evidence of non-proportionality. The graph complements Table 3, which provides the FDR-adjusted inferential results.

**Implications:** The persistence of statistically significant findings following FDR correction indicates that the observed non-proportionality cannot be attributed solely to the number of statistical comparisons. In particular, several disease-group indicators demonstrate evidence of time-dependent associations with mortality.

These findings suggest that the constant hazard ratio assumption is inappropriate for some predictors in the initial Cox model. Consequently, estimated hazard ratios should be interpreted cautiously rather than assumed to remain constant throughout the 180-day follow-up period.

**Limitations:** The FDR procedure controls the expected proportion of false discoveries among rejected hypotheses under appropriate assumptions, but it does not correct violations of the proportional hazards assumption. Furthermore, non-significant results do not establish that proportionality holds. Schoenfeld residual plots and additional model diagnostics are required to assess the nature of potential time-varying effects.

**Conclusion:** The FDR-adjusted findings provide further justification for refining the initial Cox proportional hazards model. A disease-group-stratified Cox model is a candidate approach because it allows different baseline hazard functions across disease categories. However, stratification would prevent the direct estimation of hazard ratios for disease group. Subsequent analysis should assess whether this approach adequately addresses the identified violations and improves model validity without compromising the research objectives.

In [ ]:

# STAGE 14B: FALSE DISCOVERY RATE ADJUSTMENT

from statsmodels.stats.multitest import multipletests

ph_adjusted = ph_results.copy()

reject, adjusted_p, _, _ = multipletests(
    ph_adjusted["p"],
    alpha=0.05,
    method="fdr_bh"
)

ph_adjusted["Adjusted_P"] = adjusted_p
ph_adjusted["FDR_Significant"] = reject

print("FDR-ADJUSTED PH TEST RESULTS")

display(
    ph_adjusted[
        ["test_statistic", "p",
         "Adjusted_P", "FDR_Significant"]
    ].sort_values("Adjusted_P").round(5)
)

print("\nNumber of significant predictors after FDR:")
print(reject.sum())


In [ ]:

# STAGE 14C: PUBLICATION-READY PH TEST TABLE

import pandas as pd

# Create table from actual statistical results
table3_publication = ph_adjusted[
    ["test_statistic", "p",
     "Adjusted_P", "FDR_Significant"]
].sort_values("Adjusted_P").reset_index()

table3_publication.columns = [
    "Predictor",
    "Chi-square",
    "Original p-value",
    "FDR-adjusted p-value",
    "PH violation"
]

# Convert statistical decision to readable labels
table3_publication["PH violation"] = table3_publication["PH violation"].map(
    {True: "Yes", False: "No"}
)

# Format small p-values without reporting them as zero
def format_p(p):
    return "<0.00001" if p < 0.00001 else f"{p:.5f}"

formatted_table = table3_publication.style.format({
    "Chi-square": "{:.2f}",
    "Original p-value": format_p,
    "FDR-adjusted p-value": format_p
}).hide(axis="index")

print("TABLE 3: PROPORTIONAL HAZARDS DIAGNOSTICS")
display(formatted_table)

# Save for manuscript preparation
table3_publication.to_csv(OUT_DIR / 'Table3_PH_Diagnostics_Publication.csv',
    index=False
)


**Table 3. Proportional hazards assumption test results following Benjamini–Hochberg false discovery rate adjustment.**

*Note:* Tests were based on Schoenfeld residuals from the exploratory Cox regression model. Adjusted p-values below 0.05 indicate statistical evidence of non-proportional hazards. Nonsignificant results do not establish that the proportional hazards assumption holds.


**Interpretation — Stage 14: Proportional Hazards Diagnostics and Multiple-Testing Adjustment**

The proportional hazards assumption of the exploratory Cox regression model was assessed using Schoenfeld residual-based statistical tests. To address the increased risk of false discoveries arising from multiple comparisons, the Benjamini–Hochberg false discovery rate correction was applied to the 15 coefficient-level tests.

Nine of the 15 coefficients remained statistically significant after FDR adjustment at the 5% level. These included several disease-group indicators, both cancer-status indicators, and hospital day of study entry.

The strongest statistical evidence of non-proportional hazards was observed for congestive heart failure, coma, COPD, colon cancer and lung cancer.

In contrast, six coefficients showed no statistically significant evidence of non-proportional hazards following adjustment.

**Interpretation:** These findings indicate that the relative mortality hazards associated with several clinical characteristics may change during the 180-day follow-up period. Therefore, the conventional Cox model with constant coefficients may not adequately represent all predictor effects over time.

**Methodological implication:** Further modelling should consider stratification or time-varying effects for relevant predictors. Any modifications should be justified using both statistical diagnostics and clinical considerations.

**Limitation:** Statistical significance does not establish the direction or magnitude of time variation. Moreover, failure to reject the proportional hazards assumption does not prove that the assumption is satisfied.

These findings support further investigation and revision of the Cox model before final predictive comparison with Random Survival Forests.


### Stage 15 — Disease-group-stratified Cox model


In [ ]:

# STAGE 15: STRATIFIED COX REGRESSION

from lifelines import CoxPHFitter
import pandas as pd

# Encode categorical predictors
strat_train = pd.get_dummies(
    train_df.copy(),
    columns=["sex", "ca"],
    drop_first=True,
    dtype=int
)

# Fit stratified Cox model
stratified_cox = CoxPHFitter(penalizer=0.01)

stratified_cox.fit(
    strat_train,
    duration_col="time_180",
    event_col="event_180",
    strata=["dzgroup"]
)

print("STRATIFIED COX REGRESSION RESULTS")
stratified_cox.print_summary()

print("\nTRAINING CONCORDANCE INDEX")
print(round(
    stratified_cox.concordance_index_, 4
))


**Interpretation — Stage 15: Stratified Cox Proportional Hazards Regression**

A stratified Cox proportional hazards model was developed to investigate non-proportional hazards associated with disease-group classifications in the original regression model.

The model was fitted using 7,284 patients and 3,412 observed deaths, with disease group specified as the stratification variable. This approach permitted disease-specific baseline hazard functions while estimating common regression coefficients for the remaining predictors.

Increasing age (HR ≈ 1.01), comorbidity burden (HR ≈ 1.11), and hospital day of study entry (HR ≈ 1.01) were associated with higher mortality hazards. Diabetes showed a lower adjusted hazard (HR ≈ 0.88), although this association should not be interpreted causally.

Compared with metastatic cancer, patients without cancer (HR ≈ 0.79) and those with non-metastatic cancer (HR ≈ 0.84) exhibited lower adjusted mortality hazards.

The model produced a training concordance index of 0.5997. This value should not be directly compared with the unstratified model's concordance index of 0.6719 because stratification changes the sets of comparable survival outcomes used in concordance calculations.

**Limitation:** Disease-group stratification removes the requirement for proportional baseline hazards between disease groups, but proportional hazards assumptions for the remaining covariates still require evaluation. Additionally, the model has not yet undergone held-out predictive validation.

Further Schoenfeld residual diagnostics will determine whether additional model modifications are necessary.


### Stage 16 — Stratified Cox PH diagnostics (Table 4, Figure 5)


In [ ]:

# STAGE 16: STRATIFIED COX PH DIAGNOSTICS

from lifelines.statistics import proportional_hazard_test
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Schoenfeld residual-based PH tests
strat_ph_test = proportional_hazard_test(
    stratified_cox,
    strat_train,
    time_transform="rank"
)

strat_ph = strat_ph_test.summary.copy()

# Benjamini-Hochberg FDR correction
reject, adjusted_p, _, _ = multipletests(
    strat_ph["p"],
    alpha=0.05,
    method="fdr_bh"
)

strat_ph["Adjusted_P"] = adjusted_p
strat_ph["FDR_Significant"] = reject

print("STRATIFIED COX PH TEST RESULTS")

display(
    strat_ph.sort_values(
        "Adjusted_P"
    ).round(5)
)

print("\nNumber of coefficients tested:",
      len(strat_ph))

print("FDR-significant violations:",
      reject.sum())


In [ ]:

# STAGE 16B: STRATIFIED COX PH DIAGNOSTICS
# Table 4 and graphical representation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

results16 = strat_ph.sort_values("Adjusted_P").copy()

# Create Table 4
table4 = results16[
    ["test_statistic", "p", "Adjusted_P",
     "FDR_Significant"]
].copy()

table4.columns = [
    "Chi-square", "Original p-value",
    "FDR-adjusted p-value", "PH violation"
]

table4["PH violation"] = table4[
    "PH violation"
].map({True: "Yes", False: "No"})

print("TABLE 4: STRATIFIED COX PH DIAGNOSTICS")
display(table4.round(5))

# Graph original and FDR-adjusted p-values
labels = results16.index.tolist()
y = np.arange(len(labels))

raw_scores = -np.log10(results16["p"])
fdr_scores = -np.log10(results16["Adjusted_P"])

fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    y - 0.2, raw_scores, height=0.38,
    color="#D97757", label="Original p-value"
)

ax.barh(
    y + 0.2, fdr_scores, height=0.38,
    color="#258875", label="FDR-adjusted p-value"
)

ax.axvline(
    -np.log10(0.05),
    color="gray",
    linestyle="--",
    label="5% significance threshold"
)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()

ax.set_xlabel("-log10(p-value)")
ax.set_title(
    "Figure 5. Stratified Cox Model: PH Diagnostics"
)
ax.legend()
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure05_Stratified_Cox_PH_Diagnostics.png',
    dpi=300, bbox_inches="tight"
)
plt.show()

# Export Table 4
table4.to_csv(OUT_DIR / 'Table4_Stratified_Cox_PH.csv'
)


**Interpretation — Stage 16: Proportional Hazards Diagnostics of the Stratified Cox Model**

The proportional hazards assumption of the disease-group-stratified Cox regression model was examined using Schoenfeld residual-based tests. The Benjamini–Hochberg false discovery rate adjustment was applied to eight coefficient-level tests.

None of the eight coefficients demonstrated statistically significant evidence of non-proportional hazards following FDR correction at the 5% significance level.

Age and non-metastatic cancer had the lowest adjusted p-values (both approximately 0.05595), followed by male sex (adjusted p = 0.08358). All other adjusted p-values exceeded 0.20.

Compared with the initial unstratified Cox model, which identified nine significant coefficient-level violations, the stratified model showed no remaining significant violations among the covariates tested.

These results are consistent with improved accommodation of differences in baseline hazard functions across disease groups. However, the two models involve different sets of tested coefficients, and the absence of statistically significant violations does not prove that proportional hazards holds exactly.

**Methodological implication:** The stratified Cox model is a reasonable candidate for subsequent predictive evaluation, subject to graphical residual diagnostics and assessment of model performance on the held-out test sample.

**Limitation:** The analysis does not establish that the revised model predicts mortality more accurately than the original Cox model or Random Survival Forests. Further evaluation of discrimination, calibration, and prediction error is required.


### Stage 17 — Schoenfeld residual trends (Figure 6)


In [ ]:

# STAGE 17: SCHOENFELD RESIDUAL PLOTS
# Stratified Cox model

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# Calculate scaled Schoenfeld residuals
residuals = stratified_cox.compute_residuals(
    strat_train,
    kind="scaled_schoenfeld"
)

# Focus on variables requiring closer investigation
variables = ["age", "ca_yes"]

fig, axes = plt.subplots(
    2, 1, figsize=(10, 9)
)

for ax, var in zip(axes, variables):

    # Match residuals to observed event times
    times = strat_train.loc[
        residuals.index, "time_180"
    ].to_numpy()

    values = residuals[var].to_numpy()

    valid = np.isfinite(times) & np.isfinite(values)
    times = times[valid]
    values = values[valid]

    # Smooth residual trend
    smooth = lowess(
        values, times, frac=0.25,
        return_sorted=True
    )

    ax.scatter(
        times, values,
        alpha=0.12,
        s=8,
        color="#8DA3B8"
    )

    ax.plot(
        smooth[:, 0], smooth[:, 1],
        color="#D65A4A",
        linewidth=2,
        label="LOWESS trend"
    )

    ax.axhline(
        0, color="gray",
        linestyle="--"
    )

    ax.set_title(
        f"Scaled Schoenfeld Residuals: {var}"
    )
    ax.set_xlabel("Follow-up time (days)")
    ax.set_ylabel("Scaled Schoenfeld residual")
    ax.legend()
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure06_Schoenfeld_Residuals.png',
    dpi=300,
    bbox_inches="tight"
)
plt.show()


**Interpretation — Stage 17: Graphical Assessment of the Proportional Hazards Assumption**

Scaled Schoenfeld residual plots were examined for age and non-metastatic cancer status following disease-group stratification of the Cox proportional hazards model.

For age, the LOWESS smoothed residual curve remained approximately horizontal across the 180-day follow-up period, with only minor fluctuations. This pattern was broadly consistent with a relatively stable regression coefficient over time.

For non-metastatic cancer status, the smoothed residual curve demonstrated a modest downward trend, suggesting possible time variation in the estimated association with mortality. However, substantial residual dispersion was observed, and the graphical evidence was not conclusive.

The corresponding Benjamini–Hochberg adjusted p-values were approximately 0.05595 for both predictors, exceeding the conventional 5% significance threshold.

**Interpretation:** The graphical findings, considered together with the adjusted Schoenfeld residual tests, provide no strong evidence against proportional hazards for age. Some uncertainty remains regarding the time stability of the non-metastatic cancer coefficient.

**Limitation:** The absence of statistically significant violations does not establish that proportional hazards holds exactly. Graphical smoothing may also be influenced by event-time distributions, residual variability, and the number of observations available at later follow-up times.

Overall, the stratified Cox model was retained as a candidate for predictive evaluation, with further assessment required to establish its discrimination, calibration, and generalisability.


## 5. Cox Model Evaluation

Held-out test data are used only for model evaluation. Predictions are aligned explicitly by patient index. Calibration and discrimination are examined separately.


### Stage 18A — Test-set evaluation and verified patient-index alignment (Table 5)


In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

# Binary 180-day outcome evaluation requires fully observed status at day 180.
for part, frame in [("Training",train_df),("Testing",test_df)]:
    early = int(((frame['event_180'] == 0) & (frame['time_180'] < 180)).sum())
    assert early == 0, f"{part}: {early} patients censored before day 180; use IPCW metrics."

original_test = test_cox.reindex(columns=cox_data.columns,fill_value=0)
strat_test = test_df.copy()
for column in ['sex', 'ca']:
    strat_test[column] = pd.Categorical(strat_test[column], categories=sorted(train_df[column].unique()))
strat_test = pd.get_dummies(strat_test,columns=['sex','ca'],drop_first=True,dtype=int)
strat_test = strat_test.reindex(columns=strat_train.columns,fill_value=0)

original_survival = cph.predict_survival_function(original_test, times=[180]).iloc[0]
stratified_survival = stratified_cox.predict_survival_function(strat_test, times=[180]).iloc[0]

# Guard against silent misalignment, which was found and corrected in the archive.
original_risk = (1-original_survival).reindex(test_df.index)
stratified_risk = (1-stratified_survival).reindex(test_df.index)
for label, risk in [('Original Cox',original_risk),('Stratified Cox',stratified_risk)]:
    assert risk.notna().all() and risk.between(0,1).all(), f"Invalid {label} predictions"

y_test = test_df['event_180'].astype(int)
corrected_comparison = pd.DataFrame({
    'Model':['Original Cox','Stratified Cox'],
    'ROC_AUC':[roc_auc_score(y_test,original_risk),roc_auc_score(y_test,stratified_risk)],
    'Brier_Score':[brier_score_loss(y_test,original_risk),brier_score_loss(y_test,stratified_risk)]
})
print('TABLE 5: HELD-OUT COX MODEL PERFORMANCE')
display(corrected_comparison.round(4))
corrected_comparison.to_csv(OUT_DIR/'Table05_Heldout_Cox_Performance.csv',index=False)
print('Testing patients:',len(y_test),'Observed mortality:',round(y_test.mean(),4))
print('Prediction indexes aligned:',original_risk.index.equals(y_test.index),stratified_risk.index.equals(y_test.index))


**Interpretation — Stage 18: Held-Out Predictive Performance of Cox Regression Models**

The predictive performance of the original and disease-group-stratified Cox regression models was evaluated using an independent testing subset comprising 1,821 patients.

Following verification and correction of patient-index alignment, the original Cox model achieved a receiver operating characteristic area under the curve (ROC-AUC) of 0.7124 and a Brier score of 0.2156.

The stratified Cox model achieved a comparable ROC-AUC of 0.7099 and a Brier score of 0.2164.

Both models demonstrated similar discrimination and predictive accuracy, with the original model showing marginally better numerical performance. The observed differences were small and have not yet been assessed for statistical uncertainty.

Mean predicted 180-day mortality was 47.05% for the original Cox model and 46.65% for the stratified Cox model, compared with an observed mortality proportion of 46.84%.

Both models achieved lower prediction errors than the constant-risk baseline Brier score of 0.2490.

These findings suggest that disease-group stratification substantially altered the proportional hazards diagnostics without producing a large reduction in held-out predictive discrimination.

**Limitations:** The evaluation represents internal holdout validation. Overall agreement between observed and mean predicted mortality does not establish individual-level calibration. Further assessment using calibration plots, uncertainty estimates, and additional model comparisons is necessary.


### Stage 18B — Cox ROC-AUC comparison (Figure 7)


In [ ]:

# FIGURE 6: COX MODEL ROC-AUC COMPARISON

import matplotlib.pyplot as plt

# Use verified, index-aligned test results
models = corrected_comparison["Model"]
auc_values = corrected_comparison["ROC_AUC"]

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    models,
    auc_values,
    color=["#2563EB", "#D97757"],
    width=0.55
)

# Display actual AUC values above bars
for bar, value in zip(bars, auc_values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.015,
        f"{value:.4f}",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

ax.set_ylim(0, 1)
ax.set_ylabel("ROC-AUC")
ax.set_title(
    "Figure 7. Comparison of Cox Models: 180-Day Mortality",
    fontsize=14,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure07_Cox_AUC_Comparison.png',
    dpi=300,
    bbox_inches="tight"
)
plt.show()


**Figure 7. Comparison of ROC-AUC values for the original and disease-group-stratified Cox regression models.**

The original Cox model achieved an ROC-AUC of 0.7124, compared with 0.7099 for the stratified model, using the same held-out testing sample of 1,821 patients. The difference of 0.0025 indicates very similar discrimination. Statistical uncertainty around this difference has not yet been estimated.


### Stage 18C — ROC and calibration curves (Figures 8 and 9)


In [ ]:

# STAGE 18B: ROC AND CALIBRATION CURVES

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve

y = y_test.to_numpy()

models = {
    "Original Cox": {
        "risk": original_risk.to_numpy(),
        "color": "#2563EB"
    },
    "Stratified Cox": {
        "risk": stratified_risk.to_numpy(),
        "color": "#D97757"
    }
}

# FIGURE 8: ROC CURVES
fig, ax = plt.subplots(figsize=(9, 7))

for name, info in models.items():
    fpr, tpr, _ = roc_curve(y, info["risk"])
    auc = roc_auc_score(y, info["risk"])

    ax.plot(
        fpr, tpr,
        linewidth=2.5,
        color=info["color"],
        label=f"{name} (AUC = {auc:.4f})"
    )

ax.plot(
    [0, 1], [0, 1],
    color="gray",
    linestyle="--",
    label="Random classification"
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Figure 8. ROC Curves: 180-Day Mortality Prediction")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure08_Cox_ROC.png', dpi=300)
plt.show()


# FIGURE 9: CALIBRATION CURVES
fig, ax = plt.subplots(figsize=(9, 7))

for name, info in models.items():
    observed, predicted = calibration_curve(
        y,
        info["risk"],
        n_bins=10,
        strategy="quantile"
    )

    ax.plot(
        predicted, observed,
        marker="o",
        linewidth=2,
        color=info["color"],
        label=name
    )

ax.plot(
    [0, 1], [0, 1],
    color="gray",
    linestyle="--",
    label="Perfect calibration"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel("Mean Predicted Mortality")
ax.set_ylabel("Observed Mortality Proportion")
ax.set_title("Figure 9. Calibration: 180-Day Mortality Prediction")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure09_Cox_Calibration.png', dpi=300)
plt.show()


### Figure 8. ROC curves for the original and disease-group-stratified Cox models
Interpretation: Figure 8 compares the discrimination ability of the two Cox regression models on the held-out test sample. The original Cox model achieved an ROC-AUC of 0.7124, while the stratified Cox model achieved 0.7099. Both ROC curves lie well above the diagonal reference line for random classification, showing that both models have useful discrimination. However, the difference between the two models is very small (0.0025), indicating that the original and stratified Cox models perform almost identically in distinguishing patients who died within 180 days from those who survived.


### Figure 9. Calibration curves for the original and disease-group-stratified Cox models
Interpretation: Figure 9 shows the agreement between predicted and observed 180-day mortality across risk groups. The calibration curves for both the original Cox model and the stratified Cox model lie reasonably close to the diagonal reference line representing perfect calibration. This suggests that both models provide fairly accurate probability estimates across the range of predicted risks. Minor deviations from the diagonal are present in some risk groups, but overall the two models show similar calibration performance, with no major practical difference.


### Stage 18D — Calibration by predicted-risk decile (Table 6, Figure 10)


In [ ]:

# STAGE 18C: TABLE 6 - CALIBRATION BY RISK DECILE

import pandas as pd
import numpy as np

# Patient outcomes and aligned predictions
y = test_df["event_180"].astype(int)

risk_data = {
    "Original Cox": original_risk.reindex(y.index),
    "Stratified Cox": stratified_risk.reindex(y.index)
}

calibration_rows = []

for model_name, predictions in risk_data.items():

    data = pd.DataFrame({
        "Observed": y,
        "Predicted": predictions
    })

    assert data.notna().all().all()
    assert data["Predicted"].between(0, 1).all()

    # Form approximately equal-sized risk groups
    data["Risk_Decile"] = (
        pd.qcut(
            data["Predicted"],
            q=10,
            labels=False,
            duplicates="drop"
        ) + 1
    )

    grouped = data.groupby("Risk_Decile")

    for group, subset in grouped:

        calibration_rows.append({
            "Model": model_name,
            "Risk_Decile": int(group),
            "Patients": len(subset),
            "Deaths": int(subset["Observed"].sum()),
            "Mean_Predicted": subset["Predicted"].mean(),
            "Observed_Mortality": subset["Observed"].mean()
        })

# Create calibration table
calibration_table = pd.DataFrame(calibration_rows)

calibration_table["Difference"] = (
    calibration_table["Observed_Mortality"]
    - calibration_table["Mean_Predicted"]
)

print("TABLE 6: CALIBRATION BY RISK DECILE")

display(calibration_table.round(4))

# Save original numerical precision
calibration_table.to_csv(OUT_DIR / 'Table6_Calibration_Results.csv',
    index=False
)

print("\nTable 6 generated and saved successfully.")


### Table 6. Calibration Results by Predicted-Risk Decile for the Original and Stratified Cox Regression Models

**Statistical interpretation**

Calibration was evaluated by dividing the held-out testing sample of 1,821 patients into ten approximately equal-sized groups based on predicted 180-day mortality probabilities. For each group, the mean predicted mortality, observed mortality proportion, and difference between observed and predicted outcomes were calculated.

Both models demonstrated a generally increasing pattern of observed mortality across predicted-risk deciles, indicating that patients assigned higher predicted risks tended to experience higher mortality.

However, deviations between predicted and observed mortality were evident in some risk groups.

For the original Cox model, the eighth decile exhibited the largest observed underprediction, with predicted mortality of 57.49% compared with observed mortality of 66.48%, representing a difference of approximately 8.99 percentage points.

For the stratified Cox model, the seventh decile showed an underprediction of approximately 6.02 percentage points.

In the highest-risk decile, the original Cox model predicted mortality of 84.09%, compared with observed mortality of 80.77%. The stratified Cox model predicted mortality of 79.88%, compared with observed mortality of 76.92%.

These findings suggest that both models capture broad differences in mortality risk while exhibiting imperfect calibration in certain predicted-risk groups.

**Methodological limitation:** Each risk decile contains approximately 182–183 patients. Consequently, observed mortality proportions are subject to sampling variability. The grouped calibration results alone do not establish whether differences between models are statistically meaningful.

Further assessment using calibration intercepts, calibration slopes, and uncertainty estimates is required before drawing conclusions about comparative calibration performance.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# FIGURE 10: CALIBRATION ERROR BY RISK DECILE

plot_df = calibration_table.copy()

required = ["Model", "Risk_Decile", "Difference"]
assert all(c in plot_df.columns for c in required), \
    "Check the Table 6 column names."

fig, ax = plt.subplots(figsize=(10, 6))

colors = {
    "Original Cox": "#2563EB",
    "Stratified Cox": "#F97316"
}

for model in ["Original Cox", "Stratified Cox"]:
    subset = plot_df[plot_df["Model"] == model].sort_values("Risk_Decile")

    ax.plot(
        subset["Risk_Decile"],
        subset["Difference"],
        marker="o",
        linewidth=2.5,
        markersize=6,
        label=model,
        color=colors[model]
    )

ax.axhline(0, color="black", linestyle="--",
           linewidth=1.5, label="Perfect calibration")

ax.set_xlabel("Predicted Risk Decile")
ax.set_ylabel("Observed Mortality - Predicted Mortality")
ax.set_title("Figure 10. Calibration Error by Risk Decile")
ax.set_xticks(range(1, 11))
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure10_Calibration_Error.png',
            dpi=300, bbox_inches="tight")
plt.show()


### Figure 10. Calibration Error by Risk Decile for the Original and Stratified Cox Models

**Statistical interpretation**

Figure 10 illustrates the differences between observed and predicted 180-day mortality probabilities across ten predicted-risk deciles for the original and disease-group-stratified Cox regression models.

Calibration error was calculated as observed mortality minus mean predicted mortality. Positive values indicate underprediction of mortality risk, while negative values indicate overprediction. Values closer to zero represent better agreement between predicted and observed mortality.

Both models exhibit variations in calibration error across the risk deciles. The original Cox model shows its largest positive deviation in the eighth decile (approximately 0.090), indicating an underprediction of mortality of about nine percentage points. Both models also show overprediction in some lower-risk groups, including the second decile.

Neither model consistently demonstrates smaller calibration errors across all deciles. The results suggest that although overall calibration is reasonably close, prediction accuracy varies between risk groups.

**Conclusion:** The original and stratified Cox models demonstrate broadly comparable but imperfect calibration. These results highlight the importance of assessing prediction performance across risk strata rather than relying exclusively on overall performance metrics. Statistical uncertainty around the decile-level differences has not yet been quantified.


### Stage 18E — Quantitative calibration (Table 7, Figure 11)


In [ ]:

# STAGE 18D: QUANTITATIVE CALIBRATION ASSESSMENT

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.special import logit

y = test_df["event_180"].astype(int)

models = {
    "Original Cox": original_risk.reindex(y.index),
    "Stratified Cox": stratified_risk.reindex(y.index)
}

results = []

for name, predictions in models.items():

    p = predictions.to_numpy(dtype=float)
    outcomes = y.to_numpy(dtype=int)

    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    # Prevent infinite logit values
    p_safe = np.clip(p, 1e-6, 1 - 1e-6)
    lp = logit(p_safe)

    # Calibration intercept: slope fixed at 1
    intercept_model = sm.GLM(
        outcomes,
        np.ones((len(outcomes), 1)),
        family=sm.families.Binomial(),
        offset=lp
    ).fit()

    # Calibration slope and intercept jointly
    slope_model = sm.GLM(
        outcomes,
        sm.add_constant(lp),
        family=sm.families.Binomial()
    ).fit()

    # Previously computed calibration deciles
    subset = calibration_table[
        calibration_table["Model"] == name
    ].copy()

    absolute_error = subset["Difference"].abs()

    weighted_error = np.average(
        absolute_error,
        weights=subset["Patients"]
    )

    results.append({
        "Model": name,
        "Observed_Mortality": outcomes.mean(),
        "Mean_Predicted": p.mean(),
        "Mean_Prediction_Bias": p.mean() - outcomes.mean(),
        "Calibration_Intercept": intercept_model.params[0],
        "Calibration_Slope": slope_model.params[1],
        "Slope_CI_Lower": slope_model.conf_int()[1, 0],
        "Slope_CI_Upper": slope_model.conf_int()[1, 1],
        "Weighted_Decile_Error": weighted_error,
        "Maximum_Decile_Error": absolute_error.max()
    })

table7 = pd.DataFrame(results)

print("TABLE 7: QUANTITATIVE CALIBRATION RESULTS")
display(table7.round(4))

table7.to_csv(OUT_DIR / 'Table7_Quantitative_Calibration.csv',
    index=False
)

print("Table 7 saved successfully.")


**Interpretation — Stage 18E: Quantitative Calibration Assessment**

Quantitative calibration was evaluated for the original and disease-group-stratified Cox regression models using the held-out testing sample.

The observed 180-day mortality proportion was 0.4684. Mean predicted mortality was 0.4705 for the original Cox model and 0.4665 for the stratified Cox model.

Both models demonstrated small mean prediction biases, with values of +0.0021 and −0.0019, respectively.

The calibration intercepts were −0.0098 for the original Cox model and +0.0085 for the stratified Cox model, indicating little overall systematic miscalibration.

The estimated calibration slopes were 0.9916 and 1.0507, respectively. Both estimates were close to the ideal slope of 1, suggesting reasonably calibrated predictions on average across the fitted risk scale.

**Interpretation:** The quantitative findings are consistent with broadly similar calibration performance between the two Cox regression models. However, confidence intervals and risk-group discrepancies must also be considered before making definitive claims about calibration quality.

**Limitation:** These findings are based on internal holdout validation and do not establish external generalisability or clinical usefulness.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# FIGURE 10: QUANTITATIVE CALIBRATION COMPARISON
# Locate the existing Table 7 results automatically.

required = {
    "Model", "Observed_Mortality",
    "Mean_Predicted", "Calibration_Intercept",
    "Calibration_Slope"
}

matches = [
    (name, value)
    for name, value in globals().copy().items()
    if isinstance(value, pd.DataFrame)
    and required.issubset(value.columns)
]

if not matches:
    raise ValueError(
        "Table 7 dataframe not found. "
        "Run the Table 7 calculation cell first."
    )

selected = next(
    ((n, d) for n, d in matches if n == "table7"),
    matches[-1]
)

metrics = selected[1].copy()
metrics["Model"] = (
    metrics["Model"].astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

models = ["Original Cox", "Stratified Cox"]
metrics = metrics.set_index("Model").loc[models]

colors = ["#2563EB", "#F97316"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Observed vs predicted mortality
axes[0].bar(
    models, metrics["Mean_Predicted"],
    color=colors, width=0.55
)
axes[0].axhline(
    metrics["Observed_Mortality"].iloc[0],
    color="black", linestyle="--",
    label="Observed mortality"
)
axes[0].set_title("A. Mean Predicted Mortality")
axes[0].set_ylabel("Mortality probability")
axes[0].set_ylim(0, 0.6)
axes[0].legend()

# Panel B: Calibration intercept
axes[1].bar(
    models, metrics["Calibration_Intercept"],
    color=colors, width=0.55
)
axes[1].axhline(0, color="black",
                linestyle="--")
axes[1].set_title("B. Calibration Intercept")
axes[1].set_ylabel("Intercept (ideal = 0)")

# Panel C: Calibration slope
axes[2].bar(
    models, metrics["Calibration_Slope"],
    color=colors, width=0.55
)
axes[2].axhline(
    1, color="black", linestyle="--",
    label="Ideal slope = 1"
)

# Add 95% confidence intervals when available
if {"Slope_CI_Lower", "Slope_CI_Upper"}.issubset(metrics.columns):
    lower = metrics["Slope_CI_Lower"].to_numpy(dtype=float)
    upper = metrics["Slope_CI_Upper"].to_numpy(dtype=float)
    slopes = metrics["Calibration_Slope"].to_numpy(dtype=float)

    errors = np.vstack([slopes - lower, upper - slopes])
    if np.isfinite(errors).all() and (errors >= 0).all():
        axes[2].errorbar(
            range(2), slopes,
            yerr=errors, fmt="none",
            ecolor="black", capsize=5
        )

axes[2].set_title("C. Calibration Slope")
axes[2].set_ylabel("Slope (ideal = 1)")
axes[2].set_ylim(bottom=0)
axes[2].legend()

for ax in axes:
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", rotation=15)

fig.suptitle(
    "Figure 11. Quantitative Calibration Comparison",
    fontsize=14, fontweight="bold"
)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure11_Quantitative_Calibration.png',
    dpi=300, bbox_inches="tight"
)
plt.show()


### Figure 11. Quantitative Calibration Comparison of the Original and Disease-Group-Stratified Cox Models

**Statistical Interpretation**

Figure 11 presents a quantitative comparison of the calibration performance of the original Cox proportional hazards model and the disease-group-stratified Cox regression model in predicting 180-day mortality among 1,821 patients in the held-out testing dataset. Calibration was evaluated using observed versus mean predicted mortality, calibration intercepts, and calibration slopes.

**Panel A: Mean Predicted Mortality**

The observed 180-day mortality proportion was 0.4684 (46.84%). The original Cox model produced a mean predicted mortality probability of 0.4705 (47.05%), representing an overprediction of 0.21 percentage points. In contrast, the stratified Cox model predicted 0.4665 (46.65%), corresponding to an underprediction of 0.19 percentage points.

These small differences suggest that both models estimate the overall mortality proportion reasonably accurately.

**Panel B: Calibration Intercept**

The calibration intercept measures systematic differences between predicted and observed outcomes, with an ideal value of zero.

The original Cox model produced an intercept of −0.0098, while the stratified Cox model produced an intercept of 0.0085. Both estimates are numerically close to zero, suggesting limited overall calibration-in-the-large error.

**Panel C: Calibration Slope**

The calibration slope assesses the agreement between predicted risks and observed outcomes across the prediction range. An ideal calibration slope is 1.

The original Cox model achieved a calibration slope of 0.9916, while the stratified Cox model achieved 1.0507. Both estimates are close to the ideal value, with the original Cox model having a slope numerically closer to 1.

The displayed 95% confidence intervals for both slopes include 1, indicating that neither estimate provides clear statistical evidence of departure from the ideal slope at the conventional 5% significance level.

**Overall Interpretation**

The quantitative calibration assessment indicates that both Cox regression models provide broadly comparable overall mortality predictions. The stratified Cox model demonstrates a slightly smaller absolute mean prediction bias and calibration intercept, whereas the original Cox model has a calibration slope closer to the ideal value.

However, the risk-decile analysis presented in Table 6 and Figure 9 identified variations in prediction accuracy across patient risk groups. Therefore, satisfactory overall calibration does not necessarily imply accurate predictions for every patient subgroup.

**Conclusion**

Both the original and disease-group-stratified Cox models demonstrate reasonably good overall calibration on the held-out testing dataset. Neither model can be considered statistically superior based solely on these numerical differences. Further assessment of uncertainty, discrimination, and subgroup-level performance is necessary before determining their comparative predictive reliability.


## 6. Random Survival Forest

RSF is evaluated on the **same held-out patients and event window** as the Cox models. Stage 20 was not successfully completed in the original file; the code below is prepared for a fresh run, but its results must not be claimed until verified.


### Stage 19A — RSF predictor completeness audit (Table 8, Figure 12)


In [ ]:

# STAGE 19A: RANDOM SURVIVAL FOREST DATA AUDIT
# SUPPORT2 Clinical Mortality Prediction

import numpy as np
import pandas as pd

predictors = [
    "age",
    "sex",
    "dzgroup",
    "num.co",
    "diabetes",
    "dementia",
    "ca",
    "hday"
]

outcomes = ["time_180", "event_180"]

# Ensure the expected columns exist
required = predictors + outcomes

for name, data in [
    ("Training", train_df),
    ("Testing", test_df)
]:
    missing = set(required) - set(data.columns)
    assert not missing, (
        f"{name} missing columns: {missing}"
    )

    assert data.index.is_unique, (
        f"{name} indices are not unique"
    )

    assert data["event_180"].isin([0, 1]).all()
    assert data["time_180"].between(0, 180).all()

# Confirm train-test separation
overlap = len(
    set(train_df.index) & set(test_df.index)
)

assert overlap == 0, "Patient overlap detected"

# Audit predictor missingness
audit_rows = []

for col in predictors:
    audit_rows.append({
        "Predictor": col,
        "Data_Type": str(train_df[col].dtype),
        "Training_Missing": int(
            train_df[col].isna().sum()
        ),
        "Testing_Missing": int(
            test_df[col].isna().sum()
        ),
        "Unique_Training_Values": int(
            train_df[col].nunique(dropna=True)
        )
    })

table8 = pd.DataFrame(audit_rows)

print("TABLE 8: RANDOM SURVIVAL FOREST PREDICTOR AUDIT")
display(table8)

# Check the survival outcomes
sample_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Patients": [len(train_df), len(test_df)],
    "Deaths_180": [
        int(train_df["event_180"].sum()),
        int(test_df["event_180"].sum())
    ],
    "Mortality_Percent": [
        100 * train_df["event_180"].mean(),
        100 * test_df["event_180"].mean()
    ]
})

print("\nSAMPLE AND OUTCOME VERIFICATION")
display(sample_summary.round(2))

# Check censoring before 180 days
for name, data in [
    ("Training", train_df),
    ("Testing", test_df)
]:
    early_censored = (
        (data["event_180"] == 0) &
        (data["time_180"] < 180)
    ).sum()

    print(
        f"{name} censored before day 180:",
        early_censored
    )

print("\nPatient overlap:", overlap)

# Save supporting tables
table8.to_csv(OUT_DIR / 'Table8_RSF_Predictor_Audit.csv',
    index=False
)

sample_summary.to_csv(OUT_DIR / 'RSF_Sample_Verification.csv',
    index=False
)

print("\nStage 19A verification completed.")


### Table 8. Random Survival Forest Predictor Audit

**Statistical Interpretation**

Table 8 presents the data quality assessment of eight candidate predictors selected for Random Survival Forest (RSF) modelling of 180-day mortality in the SUPPORT2 dataset.

The predictors include age, sex, primary disease group, number of comorbidities, diabetes status, dementia status, cancer status, and hospital day at study entry.

All eight predictors have complete observations in both training and testing datasets, with no missing values identified.

The categorical predictors (sex, disease group, and cancer status) require appropriate numerical encoding before model estimation. The remaining variables are already represented numerically.

**Conclusion**

The predictor audit confirms complete data availability for the selected variables. No missing-value imputation is required. However, successful data completeness checks do not independently establish predictor validity, model suitability, or the absence of information leakage.


In [ ]:

# FIGURE 12: PREDICTOR COMPLETENESS

import numpy as np
import matplotlib.pyplot as plt

predictors = [
    "age", "sex", "dzgroup", "num.co",
    "diabetes", "dementia", "ca", "hday"
]

completeness = np.array([
    [
        train_df[c].notna().mean() * 100,
        test_df[c].notna().mean() * 100
    ]
    for c in predictors
])

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(
    completeness,
    cmap="Greens",
    vmin=0,
    vmax=100,
    aspect="auto"
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Training", "Testing"])
ax.set_yticks(range(len(predictors)))
ax.set_yticklabels(predictors)

for i in range(len(predictors)):
    for j in range(2):
        ax.text(
            j, i,
            f"{completeness[i, j]:.1f}%",
            ha="center",
            va="center",
            color="white",
            fontweight="bold"
        )

ax.set_title(
    "Figure 12. RSF Predictor Data Completeness"
)

fig.colorbar(im, ax=ax, label="Completeness (%)")
plt.tight_layout()

plt.savefig(OUT_DIR / 'Figure12_Predictor_Completeness.png',
    dpi=300,
    bbox_inches="tight"
)

plt.show()


**Figure 12. Predictor Data Completeness Across Training and Testing Samples**

Figure 12 presents the percentage of complete observations for eight candidate predictors: age, sex, disease group, number of comorbidities, diabetes, dementia, cancer status, and hospital day at study entry.

All eight predictors demonstrated 100% completeness in both the training (n = 7,284) and testing (n = 1,821) datasets.

**Interpretation:** The absence of missing values eliminates the need for missing-data imputation for these selected predictors. However, completeness alone does not establish predictor validity, clinical relevance, or the absence of information leakage.


### Stage 19B — Verify 180-day survival coding (Table 9, Figure 13)


In [ ]:
# Restored Table 9 code: in the original notebook only its output remained.
validation_rows=[]
for name,subset in [('Training',train_df),('Testing',test_df)]:
    raw=survival_df.loc[subset.index]
    expected_event=((raw['death']==1)&(raw['d.time']<=180)).astype(int)
    expected_time=raw['d.time'].clip(upper=180)
    validation_rows.append({
        'Dataset':name,
        'Patients':len(subset),
        'Deaths_within_180':int(expected_event.sum()),
        'Deaths_after_180':int(((raw['death']==1)&(raw['d.time']>180)).sum()),
        'Early_Censored':int(((raw['death']==0)&(raw['d.time']<180)).sum()),
        'Event_Coding_Correct':bool(expected_event.equals(subset['event_180'])),
        'Time_Coding_Correct':bool(expected_time.equals(subset['time_180']))
    })
table9=pd.DataFrame(validation_rows)
print('TABLE 9: SURVIVAL OUTCOME VALIDATION')
display(table9)
assert table9['Event_Coding_Correct'].all()
assert table9['Time_Coding_Correct'].all()
assert (table9['Early_Censored']==0).all()
table9.to_csv(OUT_DIR/'Table09_Survival_Outcome_Validation.csv',index=False)


**Interpretation — Stage 19B: Validation of 180-Day Survival Outcomes**

The construction of the 180-day survival outcome was independently verified against the original mortality indicator and follow-up duration recorded in the SUPPORT2 dataset.

The training sample comprised 7,284 patients, including 3,412 deaths within 180 days and 1,532 deaths occurring after the 180-day observation window.

The testing sample contained 1,821 patients, including 853 deaths within 180 days and 404 deaths occurring after 180 days.

No patients were identified as censored before day 180. Both the derived event indicators and survival-time variables agreed with the original records in the training and testing samples.

These findings support the consistency of the 180-day outcome construction and confirm that the dataset is suitable for subsequent survival modelling with administrative censoring at day 180.

**Table 9.** Validation of 180-day mortality indicators, follow-up durations, and censoring status in the training and testing samples.


In [ ]:

# FIGURE 13: SURVIVAL OUTCOME DISTRIBUTION
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

required = [
    "Dataset", "Patients", "Deaths_within_180",
    "Deaths_after_180", "Early_Censored"
]

assert all(c in table9.columns for c in required), \
    "Check Table 9 column names."

data = table9.set_index("Dataset").loc[
    ["Training", "Testing"]
].copy()

total = data["Patients"].to_numpy(dtype=int)
early = data["Deaths_within_180"].to_numpy(dtype=int)
late = data["Deaths_after_180"].to_numpy(dtype=int)
other = total - early - late

assert (other >= 0).all()
assert (data["Early_Censored"] == 0).all()

counts = np.column_stack([early, late, other])
percent = counts / total[:, None] * 100

labels = [
    "Death within 180 days",
    "Death after 180 days",
    "No recorded death"
]

colors = ["#DC4C45", "#E6A23C", "#34876E"]

fig, ax = plt.subplots(figsize=(11, 5))
left = np.zeros(len(total))

for j, label in enumerate(labels):
    bars = ax.barh(
        ["Training", "Testing"],
        percent[:, j],
        left=left,
        color=colors[j],
        label=label,
        height=0.48
    )

    for i, bar in enumerate(bars):
        value = percent[i, j]
        ax.text(
            left[i] + value / 2,
            i,
            f"{counts[i,j]:,}\n({value:.1f}%)",
            ha="center", va="center",
            color="white", fontsize=10,
            fontweight="bold"
        )
    left += percent[:, j]

ax.set_xlim(0, 100)
ax.set_xlabel("Percentage of patients")
ax.set_title(
    "Figure 13. Survival Outcome Distribution\n"
    "Training vs Testing Samples"
)
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.17),
    ncol=3, frameon=False
)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure13_Survival_Outcomes.png',
    dpi=300,
    bbox_inches="tight"
)
plt.show()


### Figure 13. Survival Outcome Distribution Across Training and Testing Samples

**Statistical Interpretation**

Figure 13 illustrates the distribution of recorded mortality outcomes across the training and testing datasets used for 180-day mortality prediction.

The training sample comprised 7,284 patients, of whom 3,412 (46.84%) died within 180 days. The testing sample included 1,821 patients, with 853 deaths (46.84%) within the same period.

An additional 1,532 deaths in the training dataset and 404 deaths in the testing dataset occurred after the 180-day prediction horizon. These later deaths were not counted as mortality events within the primary prediction window.

No patients were identified as censored before day 180, and the implemented event-coding and survival-time validation checks passed for both datasets.

**Conclusion**

The training and testing samples maintain equivalent 180-day mortality proportions. This supports comparability of outcome prevalence across the two samples and provides a consistent basis for subsequent model development and evaluation.

However, these checks do not independently establish the absence of selection bias or all possible data-quality errors.


### Stage 20 — Train and evaluate Random Survival Forest (Table 10, Figure 14)


**Execution status: pending.** This is the first model-training step without a validated saved result in the original. It requires Stages 1–19 to have run during the current Colab session. Run this once and check the printed performance before writing any RSF result interpretation.


In [ ]:

# STAGE 20: RANDOM SURVIVAL FOREST TRAINING
# Install package in Colab if necessary:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# 1. Define baseline predictors
features = [
    "age", "sex", "dzgroup", "num.co",
    "diabetes", "dementia", "ca", "hday"
]

categorical = ["sex", "dzgroup", "ca"]

# Verify existing datasets and columns
assert "train_df" in globals()
assert "test_df" in globals()

required = features + ["time_180", "event_180"]

for name, data in [
    ("Training", train_df),
    ("Testing", test_df)
]:
    assert set(required).issubset(data.columns), (
        f"{name}: missing required columns"
    )
    assert not data[required].isna().any().any()
    assert data["event_180"].isin([0, 1]).all()
    assert data["time_180"].between(1, 180).all()

# 2. Encode predictors from training data only
X_train_rsf = pd.get_dummies(
    train_df[features],
    columns=categorical,
    dtype=float
)

X_test_rsf = pd.get_dummies(
    test_df[features],
    columns=categorical,
    dtype=float
).reindex(
    columns=X_train_rsf.columns,
    fill_value=0
)

# 3. Create survival outcomes
y_train_surv = Surv.from_arrays(
    event=train_df["event_180"].astype(bool).to_numpy(),
    time=train_df["time_180"].astype(float).to_numpy()
)

y_test_surv = Surv.from_arrays(
    event=test_df["event_180"].astype(bool).to_numpy(),
    time=test_df["time_180"].astype(float).to_numpy()
)

# 4. Train Random Survival Forest
rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=20,
    min_samples_leaf=20,
    max_features="sqrt",
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train_rsf, y_train_surv)

# 5. Evaluate concordance
training_cindex = rsf.score(
    X_train_rsf, y_train_surv
)

oob_cindex = rsf.oob_score_

testing_cindex = rsf.score(
    X_test_rsf, y_test_surv
)

# TABLE 10: MODEL TRAINING PERFORMANCE
rsf_table10 = pd.DataFrame({
    "Evaluation": [
        "Training",
        "Out-of-bag",
        "Held-out testing"
    ],
    "C_Index": [
        training_cindex,
        oob_cindex,
        testing_cindex
    ]
})

print("TABLE 10: RANDOM SURVIVAL FOREST PERFORMANCE")
display(rsf_table10.round(4))

print("Training patients:", len(X_train_rsf))
print("Testing patients:", len(X_test_rsf))
print("Encoded features:", X_train_rsf.shape[1])
print("Number of trees:", rsf.n_estimators)

rsf_table10.to_csv(OUT_DIR / 'Table10_RSF_Performance.csv',
    index=False
)

# FIGURE 13: CONCORDANCE COMPARISON
fig, ax = plt.subplots(figsize=(9, 5))

colors = ["#2563EB", "#E6A23C", "#159A80"]

bars = ax.bar(
    rsf_table10["Evaluation"],
    rsf_table10["C_Index"],
    color=colors,
    width=0.55
)

ax.axhline(
    0.5,
    color="gray",
    linestyle="--",
    label="Chance-level reference"
)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.02,
        f"{height:.4f}",
        ha="center",
        fontweight="bold"
    )

ax.set_ylim(0, 1)
ax.set_ylabel("Concordance Index")
ax.set_title(
    "Figure 14. Random Survival Forest\n"
    "Training, OOB and Testing Concordance"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUT_DIR / 'Figure14_RSF_Concordance.png',
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Stage 20 completed.")


### Figure 14. Random Survival Forest Training, Out-of-Bag and Testing Concordance

The Random Survival Forest model was trained using 7,284 patients and evaluated on an independent held-out testing sample of 1,821 patients. The model consisted of 100 survival trees and 18 encoded predictor features.

The training concordance index was 0.7009, indicating moderate discriminatory performance. Out-of-bag evaluation produced a concordance index of 0.6708, while the held-out testing concordance index was 0.6783.

The difference between training and testing concordance was 0.0226, suggesting a modest reduction in discrimination when the model was applied to unseen observations. The similarity between out-of-bag and testing performance provides some evidence of consistent generalization.

Overall, the Random Survival Forest demonstrated moderate survival-risk discrimination. However, additional evaluation of 180-day mortality prediction accuracy, calibration, and uncertainty is required before determining its comparative performance against the Cox regression models.

In [ ]:
# A predicted 180-day survival curve evaluated at the end of the study horizon.
# RSF step-function values are indexed on distinct observed event times.
step_times = rsf.unique_times_
position = np.searchsorted(step_times, 180, side='right') - 1
if position < 0:
    rsf_survival_180=np.ones(len(X_test_rsf))
else:
    rsf_survival_180=rsf.predict_survival_function(X_test_rsf,return_array=True)[:,position]
rsf_risk=pd.Series(1-rsf_survival_180,index=test_df.index,name='RSF_180d_risk')
assert rsf_risk.between(0,1).all() and rsf_risk.notna().all()
print('RSF day-180 mortality prediction ready for model comparison.')


## 7. Model Comparison

Only compare models on the **same test patients** using the same 180-day outcome. Concordance and ROC-AUC measure different aspects of performance and should not be treated as identical.


### Table 11 — Cox versus RSF on the held-out test set


In [ ]:
# Requires successful Stage 20 RSF training.
from sklearn.metrics import roc_auc_score, brier_score_loss
assert 'rsf' in globals() and 'rsf_risk' in globals(), "Run Stage 20 successfully first."
model_risks={
    'Original Cox':original_risk,
    'Stratified Cox':stratified_risk,
    'Random Survival Forest':rsf_risk,
}
comparison_rows=[]
for name,risk in model_risks.items():
    aligned=risk.reindex(y_test.index)
    assert aligned.notna().all() and aligned.between(0,1).all(), name
    comparison_rows.append({
        'Model':name,
        'Test_N':len(y_test),
        'ROC_AUC_180d':roc_auc_score(y_test,aligned),
        'Brier_180d':brier_score_loss(y_test,aligned)
    })
table11=pd.DataFrame(comparison_rows).sort_values('ROC_AUC_180d',ascending=False)
print('TABLE 11: HELD-OUT 180-DAY MODEL COMPARISON')
display(table11.round(4))
table11.to_csv(OUT_DIR/'Table11_Cox_vs_RSF.csv',index=False)


### Interpretation — Table 11: Comparative Predictive Performance

The predictive performance of the original Cox proportional hazards model, disease-group-stratified Cox model, and Random Survival Forest (RSF) was evaluated using the same held-out testing sample of 1,821 patients.

The Random Survival Forest achieved the highest ROC-AUC (0.7188), followed by the original Cox model (0.7124) and the stratified Cox model (0.7099). Similarly, the RSF produced the lowest Brier score (0.2136), compared with 0.2156 and 0.2164 for the original and stratified Cox models, respectively.

These findings indicate that the RSF demonstrated marginally better discrimination and prediction accuracy for 180-day mortality. However, the observed performance differences were relatively small, suggesting that the additional modelling flexibility of the RSF did not produce a substantial improvement over conventional Cox regression.

**Limitation:** Statistical uncertainty surrounding the differences has not yet been quantified. Therefore, the RSF cannot currently be considered statistically superior to the Cox models.

**Conclusion:** The Random Survival Forest demonstrated the best numerical predictive performance among the three models, although further uncertainty assessment and external validation are required before recommending it as the preferred clinical prediction model.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use actual, current Table 11 values; never hard-code performance estimates.
assert {"Model", "ROC_AUC_180d", "Brier_180d"}.issubset(table11.columns)
results = table11[["Model", "ROC_AUC_180d", "Brier_180d"]].rename(
    columns={"ROC_AUC_180d": "ROC_AUC", "Brier_180d": "Brier"}
).set_index("Model").loc[[
    "Original Cox", "Stratified Cox", "Random Survival Forest"
]].reset_index()

colors = ["#2563EB", "#F97316", "#16A34A"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, metric, heading, better in [
    (axes[0], "ROC_AUC", "A. ROC-AUC", "Higher is better"),
    (axes[1], "Brier", "B. Brier Score", "Lower is better")
]:
    bars = ax.bar(
        results["Model"], results[metric],
        color=colors, width=0.6
    )
    ax.set_ylim(0, 0.85 if metric == "ROC_AUC" else 0.27)
    ax.set_title(f"{heading}\n{better}", fontsize=12)
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.008,
            f"{bar.get_height():.4f}",
            ha="center", fontsize=10, fontweight="bold"
        )

fig.suptitle(
    "Figure 15. Held-Out Predictive Performance\n"
    "180-Day Mortality (n = 1,821)",
    fontsize=14, fontweight="bold"
)

plt.tight_layout()
plt.savefig(OUT_DIR / "Figure15_Model_Comparison.png",
            dpi=300, bbox_inches="tight")
plt.show()


### Interpretation:
The Random Survival Forest achieved the highest ROC-AUC (0.7188) and lowest Brier score (0.2136), indicating marginally better discrimination and prediction accuracy on the held-out sample of 1,821 patients. The differences between the three models are small, however, and statistical superiority has not yet been established.

Stage 21 — Bootstrap Confidence Intervals and Statistical Comparison of the Three Models.

In [ ]:

# STAGE 21: PAIRED BOOTSTRAP MODEL COMPARISON
# Original Cox vs Stratified Cox vs Random Survival Forest

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss
from IPython.display import display

# Align predictions to the same held-out patients
y = pd.Series(y_test).astype(int)

models = {
    "Original Cox": pd.Series(original_risk).reindex(y.index),
    "Stratified Cox": pd.Series(stratified_risk).reindex(y.index),
    "Random Survival Forest": pd.Series(rsf_risk).reindex(y.index)
}

assert len(y) == 1821, "Unexpected test sample size"
assert set(y.unique()) == {0, 1}
assert all(p.notna().all() for p in models.values())
assert all(((p >= 0) & (p <= 1)).all()
           for p in models.values())

# Confirm observed values match Table 11
for name, p in models.items():
    print(
        name,
        "AUC:", round(roc_auc_score(y, p), 4),
        "Brier:", round(brier_score_loss(y, p), 4)
    )

# Paired bootstrap: resample the same patients for all models
B = 2000
rng = np.random.default_rng(42)
n = len(y)

auc_results = {name: [] for name in models}
brier_results = {name: [] for name in models}

y_array = y.to_numpy()
predictions = {
    name: p.to_numpy()
    for name, p in models.items()
}

for _ in range(B):
    idx = rng.integers(0, n, n)
    y_b = y_array[idx]

    if len(np.unique(y_b)) < 2:
        continue

    for name, p in predictions.items():
        p_b = p[idx]
        auc_results[name].append(
            roc_auc_score(y_b, p_b)
        )
        brier_results[name].append(
            brier_score_loss(y_b, p_b)
        )

def ci(values):
    return np.percentile(values, [2.5, 97.5])

# Table 12: Performance estimates and bootstrap intervals
rows = []

for name in models:
    auc_low, auc_high = ci(auc_results[name])
    br_low, br_high = ci(brier_results[name])

    rows.append({
        "Model": name,
        "ROC_AUC": roc_auc_score(y_array, predictions[name]),
        "AUC_CI_Lower": auc_low,
        "AUC_CI_Upper": auc_high,
        "Brier": brier_score_loss(y_array, predictions[name]),
        "Brier_CI_Lower": br_low,
        "Brier_CI_Upper": br_high
    })

table12 = pd.DataFrame(rows)

print("\nTABLE 12: BOOTSTRAP PERFORMANCE INTERVALS")
display(table12.round(4))

# Table 13: Paired differences (RSF minus Cox)
comparison_rows = []

for competitor in ["Original Cox", "Stratified Cox"]:
    for metric, results in [
        ("ROC_AUC", auc_results),
        ("Brier", brier_results)
    ]:
        delta = (
            np.array(results["Random Survival Forest"])
            - np.array(results[competitor])
        )
        low, high = ci(delta)

        comparison_rows.append({
            "Comparison": f"RSF minus {competitor}",
            "Metric": metric,
            "Mean_Bootstrap_Difference": np.mean(delta),
            "CI_Lower": low,
            "CI_Upper": high,
            "CI_Excludes_Zero": bool(low > 0 or high < 0)
        })

table13 = pd.DataFrame(comparison_rows)

# Export full-precision numerical results for reproducibility.
table12.to_csv(OUT_DIR / "Table12_Bootstrap_Performance.csv", index=False)
table13.to_csv(OUT_DIR / "Table13_Paired_Bootstrap_Differences.csv", index=False)

print("\nTABLE 13: PAIRED MODEL DIFFERENCES")
display(table13.round(4))

# Figure 16: ROC-AUC with bootstrap confidence intervals
fig, ax = plt.subplots(figsize=(9, 5))

names = table12["Model"].tolist()
values = table12["ROC_AUC"].to_numpy()
lower = table12["AUC_CI_Lower"].to_numpy()
upper = table12["AUC_CI_Upper"].to_numpy()

ax.bar(
    names, values,
    color=["royalblue", "darkorange", "seagreen"],
    width=0.6
)

ax.errorbar(
    range(len(names)), values,
    yerr=[values - lower, upper - values],
    fmt="none", ecolor="black", capsize=6
)

ax.set_ylim(0, 1)
ax.set_ylabel("ROC-AUC")
ax.set_title(
    "Figure 16. Held-Out ROC-AUC with 95% Bootstrap Intervals"
)
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUT_DIR / "Figure16_Bootstrap_AUC.png", dpi=300, bbox_inches="tight")
plt.show()


Interpretation — Stage 21: Bootstrap Evaluation of Model Performance (Tables 12–13 and Figure 16)
The predictive performance of the Original Cox proportional hazards model, Disease-Group-Stratified Cox model, and Random Survival Forest (RSF) was compared using the same held-out testing sample of 1,821 patients. Bootstrap resampling was used to assess uncertainty in model performance and differences between models.

The Random Survival Forest achieved the highest ROC-AUC (0.7188), followed by the Original Cox model (0.7124) and Stratified Cox model (0.7099). This indicates that the RSF demonstrated slightly better discrimination between patients who died within 180 days and those who survived beyond the prediction horizon.

Similarly, the RSF achieved the lowest Brier score (0.2136), compared with 0.2156 for the Original Cox model and 0.2164 for the Stratified Cox model. Since lower Brier scores indicate better prediction accuracy, the RSF demonstrated a small numerical advantage.

However, the paired bootstrap comparisons presented in Table 13 showed that all 95% confidence intervals for the model-performance differences included zero. Therefore, the observed numerical advantages were not statistically significant at the conventional 5% level.

Figure 16 illustrates the ROC-AUC estimates and their corresponding 95% bootstrap confidence intervals. The three models demonstrated broadly similar discrimination, with overlapping uncertainty intervals.

### Stage 22 — Comparable Calibration Assessment for Cox Regression, Stratified Cox and Random Survival Forest

In [ ]:

# STAGE 22 — COMPARABLE 180-DAY CALIBRATION
# Original Cox, Stratified Cox and Random Survival Forest

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.special import logit
from sklearn.metrics import brier_score_loss

# Output folder
OUT_DIR = Path("support2_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the existing held-out outcomes and model predictions
y_cal = pd.Series(y_test).astype(int).copy()

risk_cal = {
    "Original Cox": pd.Series(original_risk).reindex(y_cal.index).astype(float),
    "Stratified Cox": pd.Series(stratified_risk).reindex(y_cal.index).astype(float),
    "Random Survival Forest": pd.Series(rsf_risk).reindex(y_cal.index).astype(float)
}

# Validate the prediction data
assert len(y_cal) == 1821, "Unexpected test sample size."
assert set(y_cal.unique()) == {0, 1}, "Outcome must contain 0 and 1."

for name, probs in risk_cal.items():
    assert probs.notna().all(), f"Missing predictions: {name}"
    assert probs.between(0, 1).all(), f"Invalid predictions: {name}"
    assert probs.index.equals(y_cal.index), f"Index mismatch: {name}"

summary_rows = []
decile_rows = []

# Calculate calibration statistics for each model
for name, probs in risk_cal.items():

    outcome = y_cal.to_numpy()
    probabilities = probs.to_numpy()

    # Clip only for numerical stability in logistic regression
    lp = logit(np.clip(probabilities, 1e-6, 1 - 1e-6))

    # Calibration-in-the-large with slope fixed at 1
    intercept_fit = sm.GLM(
        outcome,
        np.ones((len(outcome), 1)),
        family=sm.families.Binomial(),
        offset=lp
    ).fit()

    # Logistic calibration intercept and slope
    slope_fit = sm.GLM(
        outcome,
        sm.add_constant(lp),
        family=sm.families.Binomial()
    ).fit()

    # Construct risk deciles
    frame = pd.DataFrame({
        "actual": outcome,
        "predicted": probabilities
    })

    frame["decile"] = (
        pd.qcut(
            frame["predicted"].rank(method="first"),
            10,
            labels=False
        ) + 1
    )

    groups = (
        frame.groupby("decile")
        .agg(
            n=("actual", "size"),
            observed=("actual", "mean"),
            predicted=("predicted", "mean")
        )
        .reset_index()
    )

    groups["Model"] = name
    groups["absolute_error"] = abs(
        groups["observed"] - groups["predicted"]
    )

    decile_rows.append(groups)

    summary_rows.append({
        "Model": name,
        "Test_N": len(outcome),
        "Observed_180d_Mortality": outcome.mean(),
        "Mean_Predicted_180d_Mortality": probabilities.mean(),
        "Mean_Prediction_Bias": probabilities.mean() - outcome.mean(),
        "Brier_180d": brier_score_loss(outcome, probabilities),
        "Calibration_in_the_Large_Intercept": float(intercept_fit.params[0]),
        "Logistic_Calibration_Intercept": float(slope_fit.params[0]),
        "Logistic_Calibration_Slope": float(slope_fit.params[1]),
        "Weighted_Mean_Absolute_Decile_Error": float(
            np.average(
                groups["absolute_error"],
                weights=groups["n"]
            )
        )
    })

# Table 14: Summary of model calibration
calibration_summary = pd.DataFrame(summary_rows)

# Table 15: Decile-level calibration
calibration_deciles = pd.concat(
    decile_rows,
    ignore_index=True
)

# Export tables
calibration_summary.to_csv(
    OUT_DIR / "Table14_Three_Model_Calibration.csv",
    index=False
)

calibration_deciles.to_csv(
    OUT_DIR / "Table15_Three_Model_Decile_Calibration.csv",
    index=False
)

print("TABLE 14 — HELD-OUT CALIBRATION AT 180 DAYS")
display(calibration_summary.round(4))

print("\nTABLE 15 — RISK-DECILE CALIBRATION")
display(calibration_deciles.round(4))

# FIGURE 17 — Calibration curves
fig, ax = plt.subplots(figsize=(9, 7))

ax.plot(
    [0, 1], [0, 1],
    "k--",
    linewidth=1.2,
    label="Perfect calibration"
)

colors = {
    "Original Cox": "#2563eb",
    "Stratified Cox": "#ea580c",
    "Random Survival Forest": "#15803d"
}

for name in risk_cal:
    g = calibration_deciles[
        calibration_deciles["Model"] == name
    ]

    ax.plot(
        g["predicted"],
        g["observed"],
        marker="o",
        linewidth=2,
        label=name,
        color=colors[name]
    )

ax.set_xlabel("Mean Predicted 180-Day Mortality Risk")
ax.set_ylabel("Observed 180-Day Mortality Proportion")
ax.set_title(
    "Figure 17. Calibration of Three Survival Models"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.2)
ax.legend()

fig.tight_layout()

fig.savefig(
    OUT_DIR / "Figure17_Three_Model_Calibration.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nStage 22 completed.")
print("Table 14, Table 15 and Figure 17 saved to:", OUT_DIR)


### Interpretation — Stage 22: Comparative Calibration Assessment (Tables 14–15 and Figure 17)

The calibration performance of the original Cox proportional hazards model, disease-group-stratified Cox model and Random Survival Forest was evaluated using the same held-out sample of 1,821 patients.

The observed 180-day mortality proportion was 0.4684. The original Cox model predicted a mean mortality probability of 0.4705, the stratified Cox model predicted 0.4665, and the Random Survival Forest predicted 0.4686.

The Random Survival Forest demonstrated the smallest absolute mean prediction bias (0.0002), compared with 0.0021 for the original Cox model and 0.0019 for the stratified Cox model. It also achieved the lowest Brier score (0.2136), indicating a small numerical advantage in overall probability prediction accuracy.

Figure 17 shows that the three models generally followed the ideal calibration reference line, although differences between predicted and observed mortality were evident across individual risk deciles.

These findings suggest that the three models achieved broadly similar grouped calibration in the held-out sample. However, near-zero average prediction bias does not guarantee accurate predictions across the full risk distribution.

The comparative results should therefore be interpreted alongside calibration intercepts and slopes, bootstrap uncertainty, and the limitations of internal validation. The findings do not establish that Random Survival Forest provides statistically superior calibration or greater clinical utility.

**Conclusion:** Random Survival Forest demonstrated a small numerical advantage in average prediction accuracy and calibration-in-the-large, but the available evidence does not establish its overall superiority over the Cox regression models.

# 8. Research Findings, Discussion and Limitations

## 8.1 Overview of the Study

This study investigated the prediction of 180-day mortality using the SUPPORT2 dataset. Three modelling approaches were evaluated: the original Cox proportional hazards regression model, the disease-group-stratified Cox regression model, and the Random Survival Forest (RSF).

The analysis included 9,105 patients, divided into 7,284 training observations and 1,821 held-out testing observations. The observed 180-day mortality proportion was approximately 46.84% in both datasets.

Model performance was assessed using discrimination, prediction error, proportional hazards diagnostics, and bootstrap uncertainty estimation. Detailed calibration analysis was conducted for the two Cox models; equivalent RSF calibration has not yet been established.

## 8.2 Comparative Predictive Performance

The three models were evaluated using the same held-out testing sample of 1,821 patients.

| Model | ROC-AUC | Brier Score |
|---|---:|---:|
| Original Cox | 0.7124 | 0.2156 |
| Stratified Cox | 0.7099 | 0.2164 |
| Random Survival Forest | 0.7188 | 0.2136 |

The Random Survival Forest achieved the highest ROC-AUC (0.7188), indicating the best numerical discrimination between patients who experienced mortality within 180 days and those who did not.

It also achieved the lowest Brier score (0.2136), indicating slightly lower overall probability-prediction error.

However, the differences between the three models were relatively small.

## 8.3 Proportional Hazards Assumption

The initial Cox regression model exhibited evidence of proportional hazards assumption violations, motivating the development of a disease-group-stratified Cox regression model.

Following stratification, none of the eight coefficient-level Schoenfeld residual tests remained statistically significant after Benjamini–Hochberg false discovery rate adjustment at the 5% significance level.

These findings suggest that stratification provided a more suitable representation of disease-group-specific baseline hazards.

However, the absence of statistically significant violations does not prove that the proportional hazards assumption holds exactly.

### 8.4 Calibration Assessment

Calibration was evaluated for the original Cox proportional hazards model, disease-group-stratified Cox model, and Random Survival Forest using the same held-out testing sample of 1,821 patients.

The observed 180-day mortality proportion was 0.4684. The mean predicted mortality probabilities were 0.4705 for the original Cox model, 0.4665 for the stratified Cox model, and 0.4686 for the Random Survival Forest.

The corresponding mean prediction biases were +0.0021, −0.0019, and +0.0002, respectively. The Random Survival Forest demonstrated the smallest absolute average prediction bias and the lowest Brier score (0.2136), compared with 0.2156 for the original Cox model and 0.2164 for the stratified Cox model.

The previously estimated logistic calibration intercepts and slopes for the Cox models were close to their ideal values. The additional three-model calibration assessment (Table 14) provides comparable calibration estimates for the Random Survival Forest.

Figure 17 shows broadly similar grouped calibration patterns across the three models, although deviations from the ideal calibration line occur within individual risk deciles.

These findings suggest relatively limited average prediction bias in the held-out sample. However, average calibration alone does not establish accurate prediction across the entire risk distribution. The small numerical advantages observed for the Random Survival Forest should not be interpreted as evidence of statistically superior calibration without appropriate uncertainty assessment.

Overall, all three models demonstrated broadly comparable calibration patterns under internal validation, with no established evidence of clinically meaningful superiority.

## 8.5 Bootstrap Evaluation and Statistical Uncertainty

Bootstrap resampling was used to assess uncertainty in the predictive performance of the three models.

Although the Random Survival Forest achieved the highest observed ROC-AUC and lowest Brier score, the paired bootstrap comparisons showed that the 95% confidence intervals for all model-performance differences included zero.

Therefore, the numerical advantages observed for the Random Survival Forest were not statistically significant at the conventional 5% level.

These findings indicate that the available evidence does not establish statistically significant predictive superiority of the Random Survival Forest over either Cox regression model.

However, the results also do not establish statistical equivalence between the models.

## 8.6 Research Discussion

The findings illustrate an important distinction between statistical model assumptions and predictive performance.

Disease-group stratification improved the proportional hazards diagnostic results but did not improve held-out predictive discrimination or Brier score relative to the original Cox model.

Similarly, the Random Survival Forest provided modest numerical improvements in predictive performance, but these improvements were statistically inconclusive.

This suggests that increasing model flexibility or modifying modelling assumptions does not necessarily produce meaningful improvements in predictive accuracy.

Model selection should therefore consider predictive performance, statistical assumptions, calibration, uncertainty, interpretability, and intended application rather than relying on a single performance measure.

## 8.7 Study Limitations

Several limitations should be considered.

1. The analysis relied on a historical observational dataset, potentially limiting applicability to contemporary patient populations.

2. Model performance was evaluated using internal held-out testing data rather than external validation.

3. The bootstrap confidence intervals did not establish statistically significant differences between models.

4. Calibration assessments identified prediction errors across certain risk groups.

5. The analysis did not establish clinical usefulness, fairness across patient subgroups, or reliability in other healthcare settings.

6. Additional verification of the exact prediction-time origin, the clinical measurement timing and availability of all predictors, and possible information leakage is required. The code checks internal outcome consistency but does not independently establish these clinical definitions.

## 8.8 Overall Research Conclusion

The study demonstrates that the original Cox regression model, disease-group-stratified Cox regression model, and Random Survival Forest provided broadly comparable predictive performance for 180-day mortality in the SUPPORT2 dataset.

The Random Survival Forest achieved the best numerical performance, with an ROC-AUC of 0.7188 and Brier score of 0.2136. However, paired bootstrap confidence intervals did not establish statistically significant superiority over the Cox regression models.

Disease-group stratification improved the proportional hazards diagnostic findings without producing a corresponding improvement in held-out prediction performance.

Overall, the findings suggest that greater model complexity does not automatically translate into a statistically demonstrable improvement in mortality prediction.

Further external validation, comprehensive calibration evaluation, and assessment of clinical relevance are required before determining which modelling approach is most suitable for practical application.

## 9. Exported Results and Reproducibility

The tables and figures generated by this organized notebook are saved in the `support2_results/` directory inside the Colab runtime. Save the notebook to Drive and download the ZIP before disconnecting: Colab runtime files are temporary.

To regenerate everything after a runtime reset, use **Runtime → Restart session**, then **Run all**. Stop and inspect the first error if any dependency or UCI data-access issue appears. Running end to end is necessary to confirm the current environment is reproducible.


**Execution requirement:** The final archive must be regenerated only after the corrected analysis cells have been executed successfully, including Stages 20 and 21. The existence of a saved plot is not evidence of a reproducible run.


In [ ]:
import shutil
files = sorted(p for p in OUT_DIR.iterdir() if p.is_file())
print(f"Saved {len(files)} result file(s):")
for p in files:
    print('-',p.name)
archive_path = shutil.make_archive('support2_results', 'zip', root_dir=OUT_DIR)
print('ZIP created:',archive_path)
# In Colab, open the Files sidebar to download support2_results.zip.


### Audit status (review before journal submission)

Code-level corrections in this copy: Stage 13 PH diagnostic creation retained; Stage 13/14 outputs directed to the result folder; Figure 15 derives directly from Table 11; Stage 21 exports Tables 12–13 and Figure 16; duplicate conclusion and redundant post-archive export removed.

**Still outstanding:** independent fresh-runtime execution, clinical time origin and availability of predictors, data leakage assessment beyond column checks, RSF calibration, and review of modelling choices and clinical relevance. This is an audit-ready corrected copy, **not** an independently validated final analysis.


In [ ]:

from pathlib import Path
import shutil
from google.colab import files

folder = Path("/content/support2_results")
zip_path = Path("/content/support2_results.zip")

if folder.exists() and any(folder.iterdir()):
    archive = shutil.make_archive(
        "/content/support2_results",
        "zip",
        root_dir=folder
    )
    print("ZIP created:", archive)
    files.download(archive)
elif zip_path.exists():
    files.download(str(zip_path))
else:
    print("Results folder and ZIP are missing.")
    print("The Colab runtime may have restarted.")
    print("The result-generating cells must run again.")
